# C10 競賽一（玩家辨識）：測試集的每盤 embedding 與接縫核對（事前登記 02 rev 5 第 9 節，只推論）

**用途**：11/04 起的測試日用。不訓練、不調參。用 prereg02 同一份權重（prereg01 的 `best_model.pth`，只讀）和同一段程式，替三個候選池各存一份每盤 embedding：官方測試集的 **public**、**private**，以及本機驗證池 **seam_val**。seam_val 用來做接縫核對：證明這次 Colab 算出來的東西和 prereg02 相同。帶回本機後，由 `tools/t02_player.py` 在每個池各做自我檢索、閘門、防護和融合，組成 `submission_player.csv`。**這裡不打分**，Colab 上也沒有答案檔。

**對應事前登記**：`docs/PREREG_02_OPENING_FUSION.md` 第 9 節（rev 5）；實作規格 `results/test02/design/spec.md` 2.0、2.2、2.3。底稿是 `colab/prereg02_player.ipynb`，差異列在本格最後；逐格 diff 在 `results/test02/notebook_diff_player.txt`。

## 怎麼跑

1. 用**跑 prereg01、prereg02 的同一個 Google 帳號**：權重和 prereg02 的結果在那個帳號的 Drive（`MyDrive/C10/results/prereg01/`、`results/prereg02/`）。打包檔 `C10_bundle_v1.zip` 沿用，不用重傳。
2. 把 Claude 給的 `C10_test_v1.zip` 放進 `MyDrive/C10/`（檔名不要改），這份 `test02_player.ipynb` 也放進 `MyDrive/C10/`，用 Colab 開啟。
3. 「執行階段 → 變更執行階段類型」選 **G4 GPU**。G4 是灰色、點不了時，**不要換別的 GPU**：隔一段時間再試；等了一天還是拿不到，告訴 Claude（登記第 9 節第 10 條）。
4. 「執行階段 → 全部執行」，照畫面允許 Google Drive 存取。G4 約 15–20 分鐘（推估，還沒實測）。跑完前不關分頁、電腦不睡眠。
5. 最後一行印出「Drive 已寫入完成並卸載」後，zip 已經在 Drive 上：先按「執行階段 → 中斷連線並刪除執行階段」（開著會一直扣運算單元），再下載 `MyDrive/C10/results/test02/test02_player_results.zip`（彩排時在 `results/test02_rehearsal/`），把位置告訴 Claude。下面接著出現紅字時，zip 照樣下載，並把紅字截圖給 Claude，**不要自己重跑**。

詳細步驟以 `docs/COLAB_RUN.md` 的「測試集（test02）」一節為準；和這裡不一致時，以那一節為準。

## 會停下來的情況（依序）

GPU 不是 G4；prereg01 的權重或 prereg02 的 dumps 對不上（開錯帳號、檔案被動過）；找不到測試包、測試包的成員或 SHA-256 和 MANIFEST 不符；測試檔的快速格式檢查沒過；上一次用不同測試檔算的 dump 還在；Drive 剩不到 0.5 GB；任何寫入落在 prereg01、prereg02 的目錄。這些都在推論前停下。推論後：重現核對沒過（狀態 repro_failed）或唯讀目錄有變動（readonly_changed），收尾格照樣打包再停下。

## 結果寫到哪裡

`MyDrive/C10/results/test02/player/`（整個資料夾打包成 `results/test02/test02_player_results.zip`）

| 檔案 | 內容 |
|---|---|
| `<POOL>/dumps/cand_games.npz` | 候選每盤一列（候選檔的列順序）：game_id、player_id、color、emb_sum、n_windows、weights_sha256；POOL 是 public、private、seam_val |
| `<POOL>/dumps/query_games.npz` | 題目每盤一列（題目檔的列順序、slot 由小到大）：question_id、slot、color、emb_sum、n_windows、parsed |
| `<POOL>/dumps/_INPUT.json` | 這個池用的題目檔、候選檔的檔名與 SHA-256 |
| `repro_check.json`、`player_val_top5_repro.csv` | seam_val 的重現核對（和 prereg01 比，登記 2.1 的條件） |
| `seam_p02.json` | seam_val 的逐盤 embedding 和 prereg02 是否逐位元組相同（只記錄） |
| `readonly_before.json`、`readonly_after.json` | prereg01、prereg02 目錄開頭與結尾的指紋，必須相同 |
| `config.json`、`env.json`、`run_log.json`、`timings.json` | 設定（含測試包、權重、三個池的輸入檔 SHA-256）、環境、每次執行的紀錄與狀態、耗時 |

prereg01、prereg02 的目錄**只讀**：這份 notebook 任何寫入落在那裡都會被擋下。

## 中斷或出錯

- **斷線**：重新「全部執行」，整份重跑（沒有接續）。
- **preflight 停下**：照印出的中文說明處理。這時還沒開始推論，但執行階段開著就會一直扣運算單元：截圖後先按「執行階段 → 中斷連線並刪除執行階段」。
- **重現核對沒過或狀態不是 completed**：最後一格照樣產生 zip，然後停下並印出原因。下載 zip、截圖給 Claude；這一次的測試輸出不會拿去上傳（登記第 9 節第 2 條）。

## 本機冒煙測試

`colab/run_local_smoke.py` 會設 `C10_SMOKE=1`，讀同一個 `--scratch` 裡的 `results/prereg01_smoke/`、`results/prereg02_smoke/`（只讀）和 kind 為 smoke 的 `C10_test_v1.zip`，結果寫到 `results/test02_smoke/player/`。SMOKE 的縮小方式和 prereg02 冒煙測試相同，正式執行時完全不生效。

## 與 prereg02 notebook 的差異

**模型、權重、特徵、每盤 embedding 的算法、重現核對都和 prereg02 相同**；差別是每個候選池各算一次、讀測試包、多了 prereg02 的唯讀保護、測試檔的快速格式檢查和 seam_p02 紀錄。

1. **登記 9.1** 只推論，和 prereg02 相同：模型、權重（prereg01 的 `best_model.pth`，prereg02 用的同一份）、特徵、每盤 embedding 的算法都不變。準備格、官方 cell 1、環境紀錄格、官方 cell 2、官方 cell 20、重現核對格和 prereg02 逐位元組相同；每盤 embedding 格裡的 `game_embeddings()`、`save_npz()` 和候選／題目兩段迴圈（`with Pool(NPROC)` 整段）也逐位元組相同（產生本 notebook 時用 ast 逐段核對）。
2. **登記 9.4** 每個候選池各算一次每盤 embedding：同一格程式印三次，順序 public → private → seam_val（本機驗證池）。三格只差第一行 `POOL = …; QUESTIONS_CSV, CANDIDATES_CSV = POOL_FILES[POOL]`，其餘逐字相同。輸出寫到 `results/test02/player/<POOL>/dumps/`（cand_games.npz、query_games.npz，欄位和 prereg02 相同），另寫 `_INPUT.json`（題目檔、候選檔的檔名與 SHA-256）。盤數和題數依 preflight 算好的 `EXPECTED_COUNTS[POOL]` 核對（seam_val 仍是 prereg01 記的 40,000／4,935／400）。seam_val 放最後，因為 dump 格開頭會刪掉上一次的重現核對結果。
3. **登記 9.2** 接縫核對（notebook 這一層）：seam_val 用 prereg02 原封不動的重現核對格（和 prereg01 至少 398/400 題 Top-5 相同、重心最大差 ≤ 1e-4，SMOKE 全部相同），沒過時狀態記 repro_failed、照樣打包後停下。新增 `seam_p02.json`，只記錄 seam_val 的逐盤 embedding 和 prereg02 的 dumps 是否逐位元組相同（最大差、兩次的 GPU／torch／cuDNN）；判定在本機（`tools/t02_player.py`）。
4. **營運性** 測試包：讀 `MyDrive/C10/C10_test_v1.zip`。成員只能是 `tests/MANIFEST.json` 和六個官方檔名，六個都要在；MANIFEST 的 format 要是 `c10-test02 manifest v1`，kind 正式執行要是 official 或 rehearsal（SMOKE 要是 smoke），T1 的競賽一結果要是 PASS、NORMALIZED 或 FAIL_OPENING（登記第 9 節第 6 條：T1.4／T1.5 不過時競賽一照常每池程序，結果記進 config.json 的 `test_bundle.t1`；FAIL_STRUCT 或沒有結果就停下）。只解開競賽一的四個檔和 MANIFEST.json 到 `WORK/tests/`，逐檔核對 SHA-256 和 MANIFEST 相同；競賽二的兩個檔只核對檔名在不在，不讀內容。任何一項不符就停下。
5. **營運性** 快速格式檢查（T1 的一部分）：兩份題目檔各 400 題、沒有 BOM、`question_id` 不重複且前綴是 pub_／priv_、`num_games` 是 5–20 的整數、`sgf_1..20` 和 `color_1..20` 欄都在、剛好前 `num_games` 格有棋、執色是 B／W；兩份候選檔各 40,000 列＝400 位 × 各剛好 100 盤、player_id 前綴 pub_p_／priv_p_、game_id 不重複且前綴 pub_／priv_、執色是 B／W；兩池不共用玩家和棋。不符就列出來後停下。
6. **營運性** 唯讀保護擴大到 prereg02：設定格多了 `P02_ROOT`（`results/prereg02`，SMOKE 是 `prereg02_smoke`），`guard_write` 同時擋 prereg01 和 prereg02。preflight 開頭和收尾格各記一次兩個目錄的指紋（`readonly_before.json`、`readonly_after.json`，取代 `p01_readonly_*.json`），有變動時狀態記 readonly_changed。
7. **營運性** preflight 另外核對：prereg02 的 `dumps/cand_games.npz`、`query_games.npz` 的 SHA-256（正式執行時和 repo 相同；接縫的參照）、兩個檔記的 `weights_sha256` 等於 `best_model.pth` 的 SHA-256、權重是 prereg02 用的那一份；stale-dump guard：已經存在的 `<POOL>/dumps/_INPUT.json` 和這次的輸入檔不同就停下。GPU 仍然只接受 G4。
8. **營運性** 結果目錄改成 `results/test02/player/`（SMOKE：`test02_smoke`；測試包 kind 是 rehearsal：`test02_rehearsal`），結果 zip 固定叫 `test02_player_results.zip`，放在 `results/test02/`，成員從 `player/` 起算。`run_log.json` 最上層另記 `status`（running、completed、repro_failed、readonly_changed、final_check_failed、preflight_failed），和最後一筆 run 的 status 相同。
9. **營運性** config.json 的新欄位：`notebook_sha256`（本 notebook 各格原始碼的 SHA-256，算法見 `notebook_sha256_rule`）、`rehearsal`、`gpu`／`torch`／`cudnn`、`weights`、`test_bundle`（zip 與 MANIFEST 的 SHA-256、競賽一四個檔的 SHA-256）、`sets`（三個池的輸入檔 SHA-256 與列數）、`p02_inputs`、`changes_from_prereg02`（本表）。
10. **營運性** notebook：沿用的格保留 prereg02 的 cell id（三份每盤 embedding 格加上 `-<POOL>`），新增的格用 `c10-player-t02-NN`；每格的 metadata `c10` 記對應的底稿格與是否改動。notebook metadata 的 `c10` 改記底稿和本表。

In [ ]:
# 設定格（營運性）：判斷執行環境、掛載 Google Drive、決定路徑與 SMOKE 的縮小方式
import multiprocessing
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
# SMOKE 只給本機冒煙測試用（colab/run_local_smoke.py 會設 C10_SMOKE=1）；Colab 上永遠是 False
SMOKE = (not IN_COLAB) and os.environ.get('C10_SMOKE') == '1'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/C10'
    WORK = '/content/c10'
else:
    DRIVE_DIR = os.environ['C10_DRIVE_DIR']
    WORK = os.environ['C10_WORK_DIR']

# test02：輸出根目錄 results/test02；SMOKE 是 test02_smoke，測試包 MANIFEST 的 kind 是 rehearsal 時是 test02_rehearsal
# （正式、彩排、SMOKE 的結果分開放，絕不混用；kind 在 preflight 會再完整核對一次）
TEST_ZIP = os.path.join(DRIVE_DIR, 'C10_test_v1.zip')
TEST_KIND = None
if os.path.exists(TEST_ZIP):
    import json as _json
    import zipfile as _zipfile
    try:
        with _zipfile.ZipFile(TEST_ZIP) as _z:
            TEST_KIND = _json.loads(_z.read('tests/MANIFEST.json').decode('utf-8')).get('kind')
    except (KeyError, ValueError, OSError, _zipfile.BadZipFile) as _e:
        print('注意：讀不到測試包的 MANIFEST（%s: %s），preflight 會停下並說明' % (type(_e).__name__, _e))
REHEARSAL = (not SMOKE) and TEST_KIND == 'rehearsal'
RESULT_ROOT = os.path.join(DRIVE_DIR, 'results',
                           'test02_smoke' if SMOKE else 'test02_rehearsal' if REHEARSAL else 'test02')
RESULT_DIR = os.path.join(RESULT_ROOT, 'player')
# 競賽一的測試檔：每個池一份題目檔、一份候選檔（官方檔名，tutorial 359e06f README）
TEST_POOLS = {'public': ('player_identification_test_public.csv', 'player_identification_candidates_public.csv'),
              'private': ('player_identification_test_private.csv', 'player_identification_candidates_private.csv')}
BUNDLE_ZIP = os.path.join(DRIVE_DIR, 'C10_bundle_v1.zip')
BUNDLE = os.path.join(WORK, 'C10_bundle_v1')          # 打包檔解開後的目錄

# prereg01（只讀）：權重和 prereg01 的推論結果都在這裡；SMOKE 時讀本機冒煙測試產生的 prereg01_smoke
P01_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg01_smoke' if SMOKE else 'prereg01')
P01_RESULT_DIR = os.path.join(P01_ROOT, 'player')
P01_PLAYER_CKPT = os.path.join(P01_RESULT_DIR, 'ckpt')
# prereg02（只讀）：接縫核對的參照（逐盤 embedding）；SMOKE 時讀本機冒煙測試產生的 prereg02_smoke
P02_ROOT = os.path.join(DRIVE_DIR, 'results', 'prereg02_smoke' if SMOKE else 'prereg02')
P02_RESULT_DIR = os.path.join(P02_ROOT, 'player')


def _inside(path, root):
    path = os.path.normcase(os.path.realpath(path))
    root = os.path.normcase(os.path.realpath(root))
    return path == root or path.startswith(root.rstrip(os.sep) + os.sep)


def guard_write(path):
    # 所有寫檔、建資料夾、刪檔、複製、解壓、打包的目的地都先經過這裡：落在 prereg01 或 prereg02 底下就停止。
    # 回傳 path，可以直接包在呼叫裡
    for _ro, _label in ((P01_ROOT, 'prereg01'), (P02_ROOT, 'prereg02')):
        if _inside(path, _ro):
            raise RuntimeError('拒絕寫入 %s 的目錄（只讀）：%s' % (_label, path))
    return path


os.makedirs(guard_write(WORK), exist_ok=True)
os.makedirs(guard_write(RESULT_DIR), exist_ok=True)

# 官方 notebook 依賴 Linux 的 fork（worker 函式定義在 notebook 內）。
# Python 3.14 起 Linux 的預設改成 forkserver，所以在 Linux 上明確指定 fork。
if sys.platform.startswith('linux') and multiprocessing.get_start_method(allow_none=True) != 'fork':
    multiprocessing.set_start_method('fork', force=True)
if not IN_COLAB and not SMOKE and multiprocessing.get_start_method() != 'fork':
    raise SystemExit('正式執行需要 fork 啟動方式（Colab 或 Linux）；Windows／macOS 只能跑 SMOKE')

# SMOKE 的縮小方式：只縮資料量與步數，邏輯和正式一致；正式執行時下面這些值完全不會用到
SMOKE_NPROC = 2               # 特徵抽取與推論的平行數
SMOKE_N_PLAYERS = 40          # load_rows 之後，用 SEED 固定抽 40 位 ≥2 盤的玩家
SMOKE_GAMES_PER_PLAYER = 20   # 每人最多 20 盤（照資料列順序取前 20 盤）
SMOKE_BATCH_SIZE = 8
SMOKE_NUM_EPOCHS = 1
SMOKE_STEPS_PER_EPOCH = 20
SMOKE_LOG_INTERVAL = 5
SMOKE_N_QUESTIONS = 6         # 只用題目檔的前 6 題
SMOKE_Q_GAMES = 2             # 每題只用前 2 盤（CPU 推論太慢）
SMOKE_N_CAND_PLAYERS = 12     # 候選庫只用照玩家代號排序後的前 12 位
SMOKE_CAND_GAMES = 3          # 每位候選只用前 3 盤（CPU 推論太慢）

print('IN_COLAB', IN_COLAB, '| SMOKE', SMOKE, '| start method', multiprocessing.get_start_method())
print('DRIVE_DIR ', DRIVE_DIR)
print('WORK      ', WORK)
print('RESULT_DIR', RESULT_DIR, '| REHEARSAL', REHEARSAL)
print('P01_ROOT  ', P01_ROOT, '（只讀）')
print('P02_ROOT  ', P02_ROOT, '（只讀）')
print('TEST_ZIP  ', TEST_ZIP, '| kind', TEST_KIND)


In [ ]:
# 準備格（營運性）：缺套件才安裝、解開打包檔並逐檔核對 SHA-256、設定 sys.path（prereg02：不解開、不讀訓練資料）
import hashlib
import importlib
import importlib.util
import json
import shutil
import subprocess
import time
import zipfile

_t_prepare = time.time()

if IN_COLAB:
    # 版本固定成本機冒煙測試用的版本（和競賽二的 notebook 相同）
    _need = [pkg for mod, pkg in (('sgfmill', 'sgfmill==1.1.1'), ('webdataset', 'webdataset==1.0.2'))
             if importlib.util.find_spec(mod) is None]
    if _need:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _need)
        importlib.invalidate_caches()


def sha256_file(path):
    d = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            d.update(chunk)
    return d.hexdigest()


def bundle_ok():
    # 解開的目錄完整：manifest 和 Drive 上 zip 裡的一致，而且每個檔案的 SHA-256 都相符
    mpath = os.path.join(BUNDLE, 'bundle_manifest.json')
    if not os.path.exists(mpath):
        return False
    with zipfile.ZipFile(BUNDLE_ZIP) as z:
        zip_manifest = json.loads(z.read('C10_bundle_v1/bundle_manifest.json'))
    with open(mpath, encoding='utf-8') as f:
        if json.load(f) != zip_manifest:
            return False
    return all(os.path.exists(os.path.join(BUNDLE, k)) and sha256_file(os.path.join(BUNDLE, k)) == v
               for k, v in zip_manifest['files'].items())


if not os.path.exists(BUNDLE_ZIP):
    raise SystemExit('找不到 %s：請先把 C10_bundle_v1.zip 放到 Google Drive 的 MyDrive/C10/' % BUNDLE_ZIP)
if bundle_ok():
    print('打包檔已解開，逐檔 SHA-256 相符，跳過')
else:
    shutil.rmtree(guard_write(BUNDLE), ignore_errors=True)
    _local_zip = os.path.join(WORK, os.path.basename(BUNDLE_ZIP))
    shutil.copyfile(BUNDLE_ZIP, guard_write(_local_zip))
    with zipfile.ZipFile(_local_zip) as z:
        z.extractall(guard_write(WORK))
    os.remove(guard_write(_local_zip))
    if not bundle_ok():
        raise SystemExit('打包檔解開後 SHA-256 不符，停止')
    print('打包檔已解開，逐檔 SHA-256 相符')
with open(os.path.join(BUNDLE, 'bundle_manifest.json'), encoding='utf-8') as f:
    MANIFEST = json.load(f)
for k, v in MANIFEST['files'].items():
    print('  %s  %s' % (v[:16], k))
# 紀律 7：Colab 上不能有答案檔
assert not any('answers' in n for _r, _d, _ns in os.walk(BUNDLE) for n in _ns), '打包檔裡有答案檔，停止'

# 打包檔的 tools/（c10data）與 tutorial/（官方 utils、network 原檔）放到 sys.path 最前面
for _sub in ('tools', 'tutorial'):
    _p = os.path.join(BUNDLE, _sub)
    if _p not in sys.path:
        sys.path.insert(0, _p)

# prereg02：只推論，不解開、不讀訓練資料（訓練資料壓縮檔只在上面和打包檔的其他檔一起核對 SHA-256）

PREPARE_S = time.time() - _t_prepare
print('準備完成 %.1f 秒' % PREPARE_S)


In [ ]:
import glob
import gzip
import json
import os
import time
from multiprocessing import Pool, Process

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, IterableDataset

from utils import SGFParsePlayerIdentification
from network import GoPlayerResNet

print('torch', torch.__version__, '| cuda', torch.version.cuda)

In [ ]:
# 環境紀錄格（營運性）：版本、硬體、開始時間 → env.json；並開始這一次執行的 run_log／timings 紀錄
import datetime
import importlib.metadata
import platform

import network as _network_mod
import utils as _utils_mod

SOURCE_SHA256 = {  # 產生本 notebook 時 repo 裡原檔的 SHA-256
    'tools/c10data.py': '2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495',
    'tutorial/utils.py': '3bdf715c6211282be103a75275ce211fdead65703d230b21db5be6ac411c1e63',
    'tutorial/network.py': '794a0d9cc9a20984292a3746bcec6f6e106dac2ef14394cc0d142958693ec3b6',
}


def check_source(mod, key):
    # 確認模組是從打包檔載入，而且和打包檔 manifest、repo 原檔的 SHA-256 都相同；回傳 SHA-256
    _dir = os.path.abspath(os.path.join(BUNDLE, os.path.dirname(key)))
    if os.path.dirname(os.path.abspath(mod.__file__)) != _dir:
        raise SystemExit('%s 不是從打包檔 %s 載入：%s' % (mod.__name__, _dir, mod.__file__))
    _sha = sha256_file(mod.__file__)
    if _sha != MANIFEST['files'][key] or _sha != SOURCE_SHA256[key]:
        raise SystemExit('%s 的 SHA-256 %s 和打包檔 manifest（%s）或 repo 原檔（%s）不符'
                         % (key, _sha, MANIFEST['files'][key], SOURCE_SHA256[key]))
    return _sha


# 官方 cell 1 import 的 utils／network 必須是打包檔裡的官方原檔
SOURCE_CHECK = {'tutorial/utils.py': check_source(_utils_mod, 'tutorial/utils.py'),
                'tutorial/network.py': check_source(_network_mod, 'tutorial/network.py')}


def now_utc():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')


def save_json(name, obj):
    # 寫到結果目錄；json 的 float 用 repr，完整不四捨五入。先寫暫存檔再改名
    path = guard_write(os.path.join(RESULT_DIR, name))      # prereg02：先確認不在 prereg01 底下
    with open(path + '.tmp', 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(path + '.tmp', path)


def load_json(name, default):
    path = os.path.join(RESULT_DIR, name)
    if not os.path.exists(path):
        return default
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def total_ram_gb():
    try:
        import psutil
        return psutil.virtual_memory().total / 1e9
    except ImportError:
        pass
    if os.path.exists('/proc/meminfo'):
        with open('/proc/meminfo') as f:
            for line in f:
                if line.startswith('MemTotal:'):
                    return int(line.split()[1]) * 1024 / 1e9
    return None


_cuda = torch.cuda.is_available()
ENV = {
    'started_utc': now_utc(),
    'in_colab': IN_COLAB,
    'smoke': SMOKE,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if _cuda else None,
    'gpu': torch.cuda.get_device_name(0) if _cuda else None,
    'gpu_count': torch.cuda.device_count(),
    'gpu_mem_gb': torch.cuda.get_device_properties(0).total_memory / 1e9 if _cuda else None,
    'cpu_count': os.cpu_count(),
    'ram_gb': total_ram_gb(),
    'work_disk_free_gb': shutil.disk_usage(WORK).free / 1e9,
    'work_inodes_free': os.statvfs(WORK).f_favail if hasattr(os, 'statvfs') else None,
    'mp_start_method': multiprocessing.get_start_method(),
    'versions': {n: pkg_version(n) for n in ('pandas', 'numpy', 'webdataset', 'sgfmill', 'matplotlib')},
    'bundle_manifest_sha256': sha256_file(os.path.join(BUNDLE, 'bundle_manifest.json')),
    'bundle_split': MANIFEST.get('split'),
    'source_sha256': SOURCE_CHECK,           # prereg02 不讀訓練資料、不 import c10data，只核對 utils、network
}
save_json('env.json', ENV)

# run_log.json：每次執行一筆（開始時間、是否接續、每個 epoch 的步數與耗時、是否完成）
RUN_T0 = time.time()
RUN_LOG = load_json('run_log.json', {'runs': []})
for _r in RUN_LOG['runs']:
    if _r.get('status') == 'running':          # 上一次沒跑完就中斷了
        _r['status'] = 'interrupted'
RUN = {'run': len(RUN_LOG['runs']) + 1, 'started_utc': ENV['started_utc'], 'status': 'running',
       'smoke': SMOKE, 'gpu': ENV['gpu'], 'cpu_count': ENV['cpu_count']}
RUN_LOG['runs'].append(RUN)


def update_run(**kw):
    RUN.update(kw)
    RUN['seconds_so_far'] = time.time() - RUN_T0
    RUN_LOG['n_runs'] = len(RUN_LOG['runs'])
    RUN_LOG['n_resumes'] = sum(1 for r in RUN_LOG['runs'] if r.get('resumed'))
    save_json('run_log.json', RUN_LOG)


TIMINGS = load_json('timings.json', {'runs': {}})
TIMINGS['runs'][str(RUN['run'])] = {}


def record_time(stage, seconds):
    TIMINGS['runs'][str(RUN['run'])][stage] = seconds
    save_json('timings.json', TIMINGS)


update_run()
record_time('prepare_s', PREPARE_S)
print(json.dumps(ENV, ensure_ascii=False, indent=1))
print('這是第 %d 次執行' % RUN['run'])


In [ ]:
# 官方資料放置位置：training/ 與 tests/ 依官方發佈的結構擺放
# （prereg02，只推論：不讀訓練資料，所以沒有 TRAIN_DIR、SPLIT_CSV、FEAT_DIR、INDEX_PATH；
#   checkpoint 讀 prereg01 的 ckpt 目錄，只讀）
LOCAL_DIR = os.path.join(BUNDLE, 'local')     # 只有題目檔與候選庫，沒有答案
# ⑥ 題目與候選庫改讀本機驗證檔（取代官方的 TEST_DIR）
QUESTIONS_CSV = os.path.join(LOCAL_DIR, 'player_val_questions.csv')
CANDIDATES_CSV = os.path.join(LOCAL_DIR, 'player_val_candidates.csv')
CKPT_DIR = P01_PLAYER_CKPT                    # prereg01 的 ckpt 目錄（只讀；只用 best_model.pth）

NPROC = os.cpu_count()  # 特徵抽取與 SGF 解析的進程數，設成 CPU 核數（官方寫死 8；營運性）
MIN_GAMES = 2          # triplet 需要同一位玩家至少兩局（anchor 與 positive 不同局）
SEED = 42

BATCH_SIZE = 100
LEARNING_RATE = 1e-4
MARGIN = 0.5
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 15_000
LOG_INTERVAL = 5_000
TOP_N = 5              # 提交要求 Top-5

if SMOKE:
    # SMOKE 只縮小資料量與步數（數值在第一個設定格）；正式執行不會進來
    NPROC = SMOKE_NPROC
    BATCH_SIZE = SMOKE_BATCH_SIZE
    NUM_EPOCHS = SMOKE_NUM_EPOCHS
    STEPS_PER_EPOCH = SMOKE_STEPS_PER_EPOCH
    LOG_INTERVAL = SMOKE_LOG_INTERVAL

# DataLoader 的 worker 數：正式執行同官方（= NPROC）；SMOKE 且非 Colab 時為 0
# （Windows 的 spawn 無法 pickle notebook 內定義的類別；營運性）
NUM_WORKERS = 0 if (SMOKE and not IN_COLAB) else NPROC

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# prereg02：CKPT_DIR 是 prereg01 的目錄，只讀，這裡不建立資料夾
print('device', DEVICE, '| GPU', ENV['gpu'], '| NPROC', NPROC, '| DataLoader workers', NUM_WORKERS)
print('只推論、不訓練；權重讀 %s（只讀）' % os.path.join(CKPT_DIR, 'best_model.pth'))

In [ ]:
# 設定紀錄格（營運性）：推論設定、重現核對的條件、實際 NPROC、是否 SMOKE／彩排、和 prereg02 notebook 的差異 → config.json
# （prereg02 notebook 和 prereg01 的差異記在 colab/prereg02_player.ipynb 的 metadata，這裡不重抄）
T02_CHANGES = [
    '[登記 9.1] 只推論，和 prereg02 相同：模型、權重（prereg01 的 `best_model.pth`，prereg02 用的同一份）、特徵、每盤 embedding 的算法都不變。準備格、官方 cell 1、環境紀錄格、官方 cell 2、官方 cell 20、重現核對格和 prereg02 逐位元組相同；每盤 embedding 格裡的 `game_embeddings()`、`save_npz()` 和候選／題目兩段迴圈（`with Pool(NPROC)` 整段）也逐位元組相同（產生本 notebook 時用 ast 逐段核對）。',
    '[登記 9.4] 每個候選池各算一次每盤 embedding：同一格程式印三次，順序 public → private → seam_val（本機驗證池）。三格只差第一行 `POOL = …; QUESTIONS_CSV, CANDIDATES_CSV = POOL_FILES[POOL]`，其餘逐字相同。輸出寫到 `results/test02/player/<POOL>/dumps/`（cand_games.npz、query_games.npz，欄位和 prereg02 相同），另寫 `_INPUT.json`（題目檔、候選檔的檔名與 SHA-256）。盤數和題數依 preflight 算好的 `EXPECTED_COUNTS[POOL]` 核對（seam_val 仍是 prereg01 記的 40,000／4,935／400）。seam_val 放最後，因為 dump 格開頭會刪掉上一次的重現核對結果。',
    '[登記 9.2] 接縫核對（notebook 這一層）：seam_val 用 prereg02 原封不動的重現核對格（和 prereg01 至少 398/400 題 Top-5 相同、重心最大差 ≤ 1e-4，SMOKE 全部相同），沒過時狀態記 repro_failed、照樣打包後停下。新增 `seam_p02.json`，只記錄 seam_val 的逐盤 embedding 和 prereg02 的 dumps 是否逐位元組相同（最大差、兩次的 GPU／torch／cuDNN）；判定在本機（`tools/t02_player.py`）。',
    '[營運性] 測試包：讀 `MyDrive/C10/C10_test_v1.zip`。成員只能是 `tests/MANIFEST.json` 和六個官方檔名，六個都要在；MANIFEST 的 format 要是 `c10-test02 manifest v1`，kind 正式執行要是 official 或 rehearsal（SMOKE 要是 smoke），T1 的競賽一結果要是 PASS、NORMALIZED 或 FAIL_OPENING（登記第 9 節第 6 條：T1.4／T1.5 不過時競賽一照常每池程序，結果記進 config.json 的 `test_bundle.t1`；FAIL_STRUCT 或沒有結果就停下）。只解開競賽一的四個檔和 MANIFEST.json 到 `WORK/tests/`，逐檔核對 SHA-256 和 MANIFEST 相同；競賽二的兩個檔只核對檔名在不在，不讀內容。任何一項不符就停下。',
    '[營運性] 快速格式檢查（T1 的一部分）：兩份題目檔各 400 題、沒有 BOM、`question_id` 不重複且前綴是 pub_／priv_、`num_games` 是 5–20 的整數、`sgf_1..20` 和 `color_1..20` 欄都在、剛好前 `num_games` 格有棋、執色是 B／W；兩份候選檔各 40,000 列＝400 位 × 各剛好 100 盤、player_id 前綴 pub_p_／priv_p_、game_id 不重複且前綴 pub_／priv_、執色是 B／W；兩池不共用玩家和棋。不符就列出來後停下。',
    '[營運性] 唯讀保護擴大到 prereg02：設定格多了 `P02_ROOT`（`results/prereg02`，SMOKE 是 `prereg02_smoke`），`guard_write` 同時擋 prereg01 和 prereg02。preflight 開頭和收尾格各記一次兩個目錄的指紋（`readonly_before.json`、`readonly_after.json`，取代 `p01_readonly_*.json`），有變動時狀態記 readonly_changed。',
    '[營運性] preflight 另外核對：prereg02 的 `dumps/cand_games.npz`、`query_games.npz` 的 SHA-256（正式執行時和 repo 相同；接縫的參照）、兩個檔記的 `weights_sha256` 等於 `best_model.pth` 的 SHA-256、權重是 prereg02 用的那一份；stale-dump guard：已經存在的 `<POOL>/dumps/_INPUT.json` 和這次的輸入檔不同就停下。GPU 仍然只接受 G4。',
    '[營運性] 結果目錄改成 `results/test02/player/`（SMOKE：`test02_smoke`；測試包 kind 是 rehearsal：`test02_rehearsal`），結果 zip 固定叫 `test02_player_results.zip`，放在 `results/test02/`，成員從 `player/` 起算。`run_log.json` 最上層另記 `status`（running、completed、repro_failed、readonly_changed、final_check_failed、preflight_failed），和最後一筆 run 的 status 相同。',
    '[營運性] config.json 的新欄位：`notebook_sha256`（本 notebook 各格原始碼的 SHA-256，算法見 `notebook_sha256_rule`）、`rehearsal`、`gpu`／`torch`／`cudnn`、`weights`、`test_bundle`（zip 與 MANIFEST 的 SHA-256、競賽一四個檔的 SHA-256）、`sets`（三個池的輸入檔 SHA-256 與列數）、`p02_inputs`、`changes_from_prereg02`（本表）。',
    '[營運性] notebook：沿用的格保留 prereg02 的 cell id（三份每盤 embedding 格加上 `-<POOL>`），新增的格用 `c10-player-t02-NN`；每格的 metadata `c10` 記對應的底稿格與是否改動。notebook metadata 的 `c10` 改記底稿和本表。',
]
# 重現核對的條件（登記 2.1，seam_val 沿用）：Top-5 完全相同的題至少 400 − 2 = 398 題（SMOKE：全部相同）；重心最大差距 ≤ 1e-4
REPRO_MAX_MISMATCH = 0 if SMOKE else 2
REPRO_CENTROID_TOL = 1e-4
# 本 notebook 各格原始碼的 SHA-256（產生時算好；算法見 notebook_sha256_rule，本機 t02_player.py 用 repo 的 notebook 重算比對）
NB_SOURCE_SHA256 = 'fc4a2df1c9b3efaa104c4ef46e1da7183faedd34da9ae239a6083718fef202eb'
CONFIG = {
    'notebook': 'colab/test02_player.ipynb',
    'notebook_sha256': NB_SOURCE_SHA256,
    'notebook_sha256_rule': ("sha256 of json.dumps([[cell_type, source], ...], ensure_ascii=False, separators=(',', ':')) "
                             "over all cells, where the 64-hex value assigned to NB_SOURCE_SHA256 in this cell is "
                             "replaced by the placeholder @@NB_SOURCE_SHA256@@"),
    'base_notebook': 'colab/prereg02_player.ipynb',
    'base_notebook_sha256': '738c1dd81aa8a93953e45019cf6563e9b2b44ffa7cf846763c51fcc8190a5c12',
    'official_notebook': 'third_party/aicup2026_tutorial/player-identification.ipynb',
    'prereg': 'docs/PREREG_02_OPENING_FUSION.md 第 9 節（rev 5：測試日只推論、每個候選池各算一次、接縫核對）；'
              '規格 results/test02/design/spec.md 2.0、2.2、2.3',
    'split': MANIFEST.get('split'),
    'in_colab': IN_COLAB,
    'smoke': SMOKE,
    'rehearsal': REHEARSAL,
    'gpu': ENV['gpu'],
    'torch': ENV['torch'],
    'cudnn': ENV['cudnn'],
    'inference': {
        'weights': 'prereg01 的 ckpt/best_model.pth（只讀；prereg02 用的同一份，SHA-256 見 weights）',
        'model': 'GoPlayerResNet() 預設值：in_channels 17、num_channels 128、num_blocks 16、embed_dim 256',
        'features': 'SGFParsePlayerIdentification() 預設 history_length 8：8 層自己 + 8 層對手 + 1 層執色 = 17 層',
        'precision': 'fp32（和 prereg01、prereg02 的推論相同，沒有 autocast）',
        'per_game': '每盤 sgf_windows → 一盤一次 embed（GPU_BATCH 4096）→ emb_sum = e.sum(0)（float32）、n_windows = len(e)；'
                    '每次平行解析 CHUNK 256 盤（同 prereg02）',
        'pools': 'public、private（官方測試集）與 seam_val（本機驗證池，接縫核對），依序各算一次',
        'TOP_N': TOP_N,
    },
    'repro_criteria': {
        'top5_max_mismatch': REPRO_MAX_MISMATCH,
        'centroid_max_abs_diff': REPRO_CENTROID_TOL,
        'rule': 'seam_val：top_1…top_5 完全相同的題 ≥ 題數 − top5_max_mismatch；候選代號與順序相同；重心最大差距 ≤ centroid_max_abs_diff',
    },
    'nproc': NPROC,
    'device': str(DEVICE),
    'mp_start_method': multiprocessing.get_start_method(),
    'smoke_settings': ({k: v for k, v in globals().items() if k.startswith('SMOKE_')} if SMOKE else None),
    'paths': {'DRIVE_DIR': DRIVE_DIR, 'WORK': WORK, 'RESULT_ROOT': RESULT_ROOT, 'RESULT_DIR': RESULT_DIR,
              'P01_ROOT': P01_ROOT, 'P02_ROOT': P02_ROOT, 'CKPT_DIR': CKPT_DIR, 'TEST_ZIP': TEST_ZIP,
              'SEAM_QUESTIONS_CSV': QUESTIONS_CSV, 'SEAM_CANDIDATES_CSV': CANDIDATES_CSV},
    'weights': None,          # preflight 會填 {path, sha256}
    'test_bundle': None,      # preflight 會填
    'sets': None,             # preflight 會填
    'p01_inputs': None,       # preflight 會填
    'p02_inputs': None,       # preflight 會填
    'changes_from_prereg02': T02_CHANGES,
}
save_json('config.json', CONFIG)
update_run(nproc=NPROC, num_workers=NUM_WORKERS)
print('config.json 已寫入；和 prereg02 notebook 的差異 %d 項' % len(T02_CHANGES))


### preflight、唯讀指紋、測試包與快速格式檢查

開始推論前確認 prereg01 的權重與推論結果、prereg02 的 dumps 都在，而且 SHA-256 對得上（正式執行時還要和 repo 記錄的相同）；解開測試包裡競賽一的四個檔並逐檔核對 SHA-256；做快速格式檢查；確認沒有上一次用不同輸入檔留下的 dump。同時記下 prereg01、prereg02 目錄底下每個檔的大小和 mtime（`readonly_before.json`），收尾格會再記一次比對。任何一項不符就用中文說明處理方式後停下，還沒開始推論。

In [ ]:
# preflight（test02）：prereg01 的權重與推論結果、prereg02 的 dumps、測試包都對得上，快速格式檢查也過，才開始推論；
# 並記下 prereg01、prereg02 目錄開頭的指紋
import csv
import io
from collections import Counter

_t_pre = time.time()
P01_FILES = {
    'best_model.pth': os.path.join(CKPT_DIR, 'best_model.pth'),
    'player_val_top5.csv': os.path.join(P01_RESULT_DIR, 'player_val_top5.csv'),
    'player_val_embeddings.npz': os.path.join(P01_RESULT_DIR, 'player_val_embeddings.npz'),
    'predict_summary.json': os.path.join(P01_RESULT_DIR, 'predict_summary.json'),
}
# 正式執行時必須和 repo 的 results/prereg01/player/ 相同（產生本 notebook 時從 repo 算出來寫進這裡）；SMOKE 不比
P01_EXPECTED_SHA256 = {  # repo 的 results/prereg01/player/ 算出來的 SHA-256
    'best_model.pth': '356eb4f65488925e2a552ea382979780c903ef515b85d329a3805db837b7773b',
    'player_val_top5.csv': '0f5761a11b9f68ab07b2bc91b9650236f85b8cbeff45d459753dbd24459d3ec7',
    'player_val_embeddings.npz': 'c247b2653e73589975d558b17e96c4a1f38c3e4d66468d56ea6d02596e15752d',
    'predict_summary.json': '88b720242fef538a99327fdbdf839515c214a8492e2bf551f48b93c4c81eeb32',
}
P01_EXPECTED_COUNTS = {  # repo 的 predict_summary.json 記的題數、盤數
    'n_questions': 400,
    'n_query_games': 4935,
    'n_candidate_rows': 40000,
    'n_candidate_players': 400,
    'n_centroids': 400,
    'n_questions_embedded': 400,
    'n_fallback_questions': 0,
}
MIN_DRIVE_FREE_GB = 0.5
GPU_MODEL_REQUIRED = 'RTX PRO 6000'   # G4；prereg01 也是在它上面跑的（登記第 7 節第 8 條）


def ro_fingerprint(root):
    # 唯讀目錄底下每個檔的相對路徑、大小、mtime（只看 stat，不讀內容）
    files = {}
    for _root, _dirs, _files in os.walk(root):
        _dirs.sort()
        for _f in sorted(_files):
            _p = os.path.join(_root, _f)
            _st = os.stat(_p)
            files[os.path.relpath(_p, root).replace(os.sep, '/')] = {'size': _st.st_size,
                                                                      'mtime_ns': _st.st_mtime_ns}
    return {'root': root, 'taken_utc': now_utc(), 'n_files': len(files), 'files': files}


def readonly_fingerprints():
    # test02：prereg01、prereg02 兩個唯讀目錄的指紋
    return {'prereg01': ro_fingerprint(P01_ROOT), 'prereg02': ro_fingerprint(P02_ROOT)}


def set_status(status, **kw):
    # test02：run_log.json 最上層也記這次執行的狀態（本機工具讀它）；每筆 run 的 status 照舊
    RUN_LOG['status'] = status
    update_run(status=status, **kw)


def preflight_stop(msg):
    set_status('preflight_failed', preflight_error=msg)
    raise SystemExit('preflight 沒過，還沒開始推論：\n' + msg)


set_status('running')


_ACCOUNT_HINT = ('處理方式：確認這個 Colab 用的是跑 prereg01、prereg02 的同一個 Google 帳號（權重和結果在那個帳號的 '
                 'Drive），而且 MyDrive/C10/results/prereg01/、results/prereg02/ 沒有被改名、搬動或重跑過。不要自己改檔；'
                 '還是不對就截圖給 Claude。')
for _ro, _label in ((P01_ROOT, 'prereg01'), (P02_ROOT, 'prereg02')):
    if not os.path.isdir(_ro):
        preflight_stop('找不到 %s 的結果目錄 %s。\n%s' % (_label, _ro, _ACCOUNT_HINT))
READONLY_BEFORE = readonly_fingerprints()
save_json('readonly_before.json', READONLY_BEFORE)
print('唯讀目錄指紋（開頭）：prereg01 %d 個檔、prereg02 %d 個檔'
      % (READONLY_BEFORE['prereg01']['n_files'], READONLY_BEFORE['prereg02']['n_files']))

_missing = [k for k, p in P01_FILES.items() if not os.path.exists(p)]
if _missing:
    preflight_stop('找不到 prereg01 的 %s（在 %s）。\n%s' % ('、'.join(_missing), P01_RESULT_DIR, _ACCOUNT_HINT))
P01_SHA256 = {k: sha256_file(p) for k, p in P01_FILES.items()}
with open(P01_FILES['predict_summary.json'], encoding='utf-8') as f:
    P01_SUMMARY = json.load(f)
P01_ENV = None
if os.path.exists(os.path.join(P01_RESULT_DIR, 'env.json')):
    with open(os.path.join(P01_RESULT_DIR, 'env.json'), encoding='utf-8') as f:
        P01_ENV = json.load(f)
QUESTIONS_SHA256 = sha256_file(QUESTIONS_CSV)
CANDIDATES_SHA256 = sha256_file(CANDIDATES_CSV)

_problems = []
_gpu_problems = []            # GPU 的問題不附帳號的處理方式
if P01_SUMMARY.get('smoke') != SMOKE:
    _problems.append('prereg01 的 predict_summary.json 記 smoke=%r，這次是 smoke=%r' % (P01_SUMMARY.get('smoke'), SMOKE))
if P01_SUMMARY.get('weights_sha256') != P01_SHA256['best_model.pth']:
    _problems.append('best_model.pth 的 SHA-256 %s 和 prereg01 推論時用的 %s 不同'
                     % (P01_SHA256['best_model.pth'], P01_SUMMARY.get('weights_sha256')))
if P01_SUMMARY.get('output_sha256') != P01_SHA256['player_val_top5.csv']:
    _problems.append('player_val_top5.csv 的 SHA-256 %s 和 prereg01 寫出時的 %s 不同'
                     % (P01_SHA256['player_val_top5.csv'], P01_SUMMARY.get('output_sha256')))
if P01_SUMMARY.get('questions_csv_sha256') != QUESTIONS_SHA256:
    _problems.append('題目檔的 SHA-256 和 prereg01 推論時用的不同')
if P01_SUMMARY.get('candidates_csv_sha256') != CANDIDATES_SHA256:
    _problems.append('候選檔的 SHA-256 和 prereg01 推論時用的不同')
if not SMOKE:
    for _k, _v in P01_EXPECTED_SHA256.items():
        if P01_SHA256[_k] != _v:
            _problems.append('%s 的 SHA-256 %s 和 repo 記錄的 %s 不同' % (_k, P01_SHA256[_k], _v))
    for _k, _v in P01_EXPECTED_COUNTS.items():
        if P01_SUMMARY.get(_k) != _v:
            _problems.append('predict_summary.json 的 %s = %r，repo 記錄的是 %r' % (_k, P01_SUMMARY.get(_k), _v))
    # GPU：只用 G4（RTX PRO 6000）。重現核對不因 GPU 放寬，拿不到 G4 時先等，不換 GPU（登記第 7 節第 8 條）。
    # Colab 在選的 GPU 沒空時可能改派別的 GPU，所以核對型號（環境紀錄格記的 ENV['gpu']）
    if not torch.cuda.is_available():
        _gpu_problems.append('沒有 GPU。處理方式：「執行階段 → 變更執行階段類型」選 G4 GPU → 儲存，再重新「全部執行」'
                             '（G4 是灰色時先不要換別的 GPU，隔一段時間再試，見第一格）')
    elif GPU_MODEL_REQUIRED not in (ENV['gpu'] or ''):
        _gpu_problems.append('這次拿到的 GPU 是 %s，不是 G4（RTX PRO 6000）。競賽一的重現核對不因 GPU 放寬，所以不在別的 GPU '
                             '上跑。處理方式：「執行階段 → 中斷連線並刪除執行階段」，再「執行階段 → 變更執行階段類型」選 G4 GPU '
                             '→ 儲存，重新「全部執行」；G4 是灰色時先不要換別的 GPU，隔一段時間再試，等了一天還拿不到再告訴 Claude'
                             % ENV['gpu'])
try:
    _free_gb = shutil.disk_usage(DRIVE_DIR).free / 1e9
except OSError as e:          # 查不到就只提醒（同 prereg02 rank 的 preflight）
    _free_gb = None
    print('注意：查不到 Drive 剩餘空間（%s: %s），請自己確認還有 %.1f GB 以上' % (type(e).__name__, e, MIN_DRIVE_FREE_GB))
if _free_gb is not None and _free_gb < MIN_DRIVE_FREE_GB:
    _problems.append('Google Drive 只剩 %.2f GB（至少要 %.1f GB）。處理方式：清出空間後重新「全部執行」'
                     % (_free_gb, MIN_DRIVE_FREE_GB))
if _problems or _gpu_problems:
    preflight_stop('\n'.join('  - ' + p for p in _problems + _gpu_problems)
                   + ('\n' + _ACCOUNT_HINT if _problems else ''))

# ---- test02：prereg02 的 dumps（接縫的參照）與權重
WEIGHTS_SHA256_PIN = '356eb4f65488925e2a552ea382979780c903ef515b85d329a3805db837b7773b'   # prereg02 frozen_config.json 記的權重（= prereg01 的 best_model.pth）
P02_DUMP_SHA256 = {'cand_games.npz': '56ed03d12e9a79871a21b5f69dc4bd4bb8c8aa69b1544cb8ec799e80227cfdea', 'query_games.npz': '665d0888a93f6632eb19edac76abf996d5499cd27c683d9844b97cb274b6f689'}
P02_DUMPS = {k: os.path.join(P02_RESULT_DIR, 'dumps', k) for k in ('cand_games.npz', 'query_games.npz')}
_t02 = []
if not SMOKE and P01_SHA256['best_model.pth'] != WEIGHTS_SHA256_PIN:
    _t02.append('best_model.pth 的 SHA-256 %s 不是 prereg02 用的 %s' % (P01_SHA256['best_model.pth'], WEIGHTS_SHA256_PIN))
P02_SHA256 = {}
for _k, _p in P02_DUMPS.items():
    if not os.path.exists(_p):
        _t02.append('找不到 prereg02 的 dumps/%s（在 %s）' % (_k, P02_RESULT_DIR))
        continue
    P02_SHA256[_k] = sha256_file(_p)
    if not SMOKE and P02_SHA256[_k] != P02_DUMP_SHA256[_k]:
        _t02.append('prereg02 的 dumps/%s 的 SHA-256 %s 和 repo 記錄的 %s 不同' % (_k, P02_SHA256[_k], P02_DUMP_SHA256[_k]))
    with np.load(_p) as _z:
        _w = str(_z['weights_sha256']) if 'weights_sha256' in _z.files else None
    if _w != P01_SHA256['best_model.pth']:
        _t02.append('prereg02 的 dumps/%s 記的權重 %s 和 best_model.pth 的 %s 不同' % (_k, _w, P01_SHA256['best_model.pth']))
P02_ENV = None
if os.path.exists(os.path.join(P02_RESULT_DIR, 'env.json')):
    with open(os.path.join(P02_RESULT_DIR, 'env.json'), encoding='utf-8') as f:
        P02_ENV = json.load(f)
if _t02:
    preflight_stop('\n'.join('  - ' + p for p in _t02) + '\n' + _ACCOUNT_HINT)

# ---- test02：測試包 C10_test_v1.zip（成員、MANIFEST、SHA-256）；只解開競賽一的四個檔
TEST_NAMES = ['rank_prediction_test_public.csv', 'rank_prediction_test_private.csv', 'player_identification_test_public.csv', 'player_identification_test_private.csv', 'player_identification_candidates_public.csv', 'player_identification_candidates_private.csv']
PLAYER_NAMES = [n for _pool in ('public', 'private') for n in TEST_POOLS[_pool]]
TESTS_DIR = os.path.join(WORK, 'tests')
_ZIP_HINT = ('處理方式：不要自己重傳、解壓或改名。看一下 MyDrive/C10/ 裡是不是剛好一個 C10_test_v1.zip'
             '（不是 C10_test_v1 (1).zip，也不是解壓出來的資料夾），把這段訊息截圖給 Claude。')
if not os.path.exists(TEST_ZIP):
    preflight_stop('找不到測試包 %s。\n%s' % (TEST_ZIP, _ZIP_HINT))
TEST_ZIP_SHA256 = sha256_file(TEST_ZIP)
_zp = []
with zipfile.ZipFile(TEST_ZIP) as _z:
    _names = _z.namelist()
    _allowed = {'tests/MANIFEST.json'} | {'tests/' + n for n in TEST_NAMES}
    _odd = sorted(n for n in set(_names) if n not in _allowed)
    if _odd:
        _zp.append('測試包裡有不該有的成員：%s' % _odd[:5])
    if len(_names) != len(set(_names)):
        _zp.append('測試包裡有重複的成員')
    _missing = sorted(n for n in _allowed if n not in _names)
    if _missing:
        _zp.append('測試包缺少：%s' % _missing)
    TEST_MANIFEST_BYTES = _z.read('tests/MANIFEST.json') if 'tests/MANIFEST.json' in _names else b''
if _zp:
    preflight_stop('\n'.join('  - ' + p for p in _zp) + '\n' + _ZIP_HINT)
try:
    TEST_MANIFEST = json.loads(TEST_MANIFEST_BYTES.decode('utf-8'))
except ValueError as e:
    preflight_stop('測試包的 MANIFEST.json 讀不懂（%s）。\n%s' % (e, _ZIP_HINT))
_kinds = ('smoke',) if SMOKE else ('official', 'rehearsal')
if TEST_MANIFEST.get('format') != 'c10-test02 manifest v1':
    _zp.append('MANIFEST 的 format 是 %r，不是 c10-test02 manifest v1' % TEST_MANIFEST.get('format'))
if TEST_MANIFEST.get('kind') not in _kinds:
    _zp.append('MANIFEST 的 kind 是 %r（這次只接受 %s）' % (TEST_MANIFEST.get('kind'), '、'.join(_kinds)))
if (TEST_MANIFEST.get('kind') == 'rehearsal') != REHEARSAL:
    _zp.append('MANIFEST 的 kind %r 和設定格判斷的不一致（REHEARSAL=%r）' % (TEST_MANIFEST.get('kind'), REHEARSAL))
_mfiles = TEST_MANIFEST.get('files') or {}
if sorted(_mfiles) != sorted(TEST_NAMES):
    _zp.append('MANIFEST 記的檔案 %s 不是官方的六個檔名' % sorted(_mfiles))
# 登記第 9 節第 6 條：FAIL_OPENING（T1.4／T1.5 不過）時競賽一照常每池程序；FAIL_STRUCT 或沒有結果就停
if (TEST_MANIFEST.get('t1') or {}).get('player') not in ('PASS', 'NORMALIZED', 'FAIL_OPENING'):
    _zp.append('MANIFEST 記的競賽一 T1 結果是 %r（要是 PASS、NORMALIZED 或 FAIL_OPENING）' % (TEST_MANIFEST.get('t1') or {}).get('player'))
if _zp:
    preflight_stop('\n'.join('  - ' + p for p in _zp) + '\n' + _ZIP_HINT)
# 競賽二的兩個檔只核對檔名在不在（上面），不解開、不讀內容
if os.path.isdir(TESTS_DIR):
    shutil.rmtree(guard_write(TESTS_DIR))
with zipfile.ZipFile(TEST_ZIP) as _z:
    for _n in ['tests/MANIFEST.json'] + ['tests/' + n for n in PLAYER_NAMES]:
        _z.extract(_n, guard_write(WORK))
TEST_SHA256 = {}
for _n in PLAYER_NAMES:
    _p = os.path.join(TESTS_DIR, _n)
    TEST_SHA256[_n] = sha256_file(_p)
    if TEST_SHA256[_n] != (_mfiles.get(_n) or {}).get('sha256'):
        _zp.append('%s 的 SHA-256 %s 和 MANIFEST 記的 %s 不同' % (_n, TEST_SHA256[_n], (_mfiles.get(_n) or {}).get('sha256')))
    if 'bytes' in (_mfiles.get(_n) or {}) and os.path.getsize(_p) != _mfiles[_n]['bytes']:
        _zp.append('%s 有 %d bytes，MANIFEST 記 %s' % (_n, os.path.getsize(_p), _mfiles[_n]['bytes']))
with open(os.path.join(TESTS_DIR, 'MANIFEST.json'), 'rb') as f:
    _mb = f.read()
TEST_MANIFEST_SHA256 = hashlib.sha256(_mb).hexdigest()
if _mb != TEST_MANIFEST_BYTES:
    _zp.append('解開後的 MANIFEST.json 和 zip 裡的不同')
if _zp:
    preflight_stop('\n'.join('  - ' + p for p in _zp) + '\n' + _ZIP_HINT)

# ---- test02：快速格式檢查（T1 的一部分）
csv.field_size_limit(10 ** 9)
QCOLS = ['question_id', 'num_games'] + ['sgf_%d' % i for i in range(1, 21)] + ['color_%d' % i for i in range(1, 21)]
CCOLS = ['game_id', 'player_id', 'sgf_content', 'color']
PREFIXES = {'public': ('pub_', 'pub_p_', 'pub_'), 'private': ('priv_', 'priv_p_', 'priv_')}   # 題目、玩家、棋


def read_test_csv(path):
    with open(path, 'rb') as f:
        raw = f.read()
    rd = csv.DictReader(io.StringIO(raw.decode('utf-8-sig'), newline=''))
    rows = list(rd)
    return raw.startswith(b'\xef\xbb\xbf'), list(rd.fieldnames or []), rows


def quick_questions(name, prefix, problems):
    bom, cols, rows = read_test_csv(os.path.join(TESTS_DIR, name))
    if bom:
        problems.append('%s：開頭有 BOM' % name)
    miss = [c for c in QCOLS if c not in cols]
    if miss:
        problems.append('%s：缺欄位 %s' % (name, miss[:5]))
        return None
    if len(rows) != 400:
        problems.append('%s：%d 題，不是 400 題' % (name, len(rows)))
    ids = [r['question_id'] for r in rows]
    if len(set(ids)) != len(ids):
        problems.append('%s：question_id 有重複' % name)
    _bad = [q for q in ids if not q.startswith(prefix)]
    if _bad:
        problems.append('%s：%d 題的 question_id 不是 %s 開頭，例如 %s' % (name, len(_bad), prefix, _bad[:3]))
    bad, num_games = [], []
    for r in rows:
        try:
            n = int(r['num_games'])
        except (TypeError, ValueError):
            bad.append('%s 的 num_games=%r' % (r['question_id'], r['num_games']))
            continue
        if not 5 <= n <= 20:
            bad.append('%s 的 num_games=%d' % (r['question_id'], n))
            continue
        filled = [i for i in range(1, 21) if (r['sgf_%d' % i] or '').strip()]
        if filled != list(range(1, n + 1)):
            bad.append('%s 有棋的是第 %s 格，num_games=%d' % (r['question_id'], filled, n))
        if any(r['color_%d' % i] not in ('B', 'W') for i in range(1, n + 1)):
            bad.append('%s 的執色不是 B／W' % r['question_id'])
        num_games.append(n)
    if bad:
        problems.append('%s：%d 題格式不對，例如 %s' % (name, len(bad), bad[:3]))
    return {'rows': len(rows), 'num_games': num_games}


def quick_candidates(name, pprefix, gprefix, problems):
    bom, cols, rows = read_test_csv(os.path.join(TESTS_DIR, name))
    if bom:
        problems.append('%s：開頭有 BOM' % name)
    miss = [c for c in CCOLS if c not in cols]
    if miss:
        problems.append('%s：缺欄位 %s' % (name, miss))
        return None
    if len(rows) != 40000:
        problems.append('%s：%d 列，不是 40,000 列' % (name, len(rows)))
    per = Counter(r['player_id'] for r in rows)
    if len(per) != 400 or set(per.values()) != {100}:
        problems.append('%s：%d 位候選，每人盤數 %s（要 400 位 × 各剛好 100 盤；登記第 9 節第 4 條：不是的話停下）'
                        % (name, len(per), sorted(set(per.values()))[:6]))
    _bad = [p for p in per if not p.startswith(pprefix)]
    if _bad:
        problems.append('%s：%d 位候選的 player_id 不是 %s 開頭，例如 %s' % (name, len(_bad), pprefix, _bad[:3]))
    gids = [r['game_id'] for r in rows]
    if len(set(gids)) != len(gids):
        problems.append('%s：game_id 有重複' % name)
    _bad = [g for g in gids if not g.startswith(gprefix)]
    if _bad:
        problems.append('%s：%d 盤的 game_id 不是 %s 開頭，例如 %s' % (name, len(_bad), gprefix, _bad[:3]))
    _bad = [r['game_id'] for r in rows if r['color'] not in ('B', 'W')]
    if _bad:
        problems.append('%s：%d 盤的執色不是 B／W，例如 %s' % (name, len(_bad), _bad[:3]))
    return {'rows': len(rows), 'players': set(per), 'games': set(gids), 'per_player': per}


_qp = []
QUICK = {}
for _pool, (_qn, _cn) in TEST_POOLS.items():
    QUICK[_pool] = {'q': quick_questions(_qn, PREFIXES[_pool][0], _qp),
                    'c': quick_candidates(_cn, PREFIXES[_pool][1], PREFIXES[_pool][2], _qp)}
if QUICK['public']['c'] and QUICK['private']['c']:
    if QUICK['public']['c']['players'] & QUICK['private']['c']['players']:
        _qp.append('public 和 private 的候選有共用的 player_id')
    if QUICK['public']['c']['games'] & QUICK['private']['c']['games']:
        _qp.append('public 和 private 的候選有共用的 game_id')
if _qp:
    preflight_stop('測試檔的快速格式檢查沒過（這時還沒開始推論）：\n' + '\n'.join('  - ' + p for p in _qp)
                   + '\n處理方式：不要自己改檔，截圖給 Claude。')


def smoke_counts(qinfo, cinfo):
    # SMOKE 的縮小方式和 dump 格相同：候選照代號排序的前 SMOKE_N_CAND_PLAYERS 位、每人前 SMOKE_CAND_GAMES 盤；
    # 題目檔前 SMOKE_N_QUESTIONS 題、每題前 SMOKE_Q_GAMES 盤
    keep = sorted(cinfo['per_player'])[:SMOKE_N_CAND_PLAYERS]
    return (sum(min(cinfo['per_player'][p], SMOKE_CAND_GAMES) for p in keep),
            sum(min(n, SMOKE_Q_GAMES) for n in qinfo['num_games'][:SMOKE_N_QUESTIONS]),
            min(qinfo['rows'], SMOKE_N_QUESTIONS))


# 每個池的 (候選盤數, 題目盤數, 題數)；dump 格照這個核對。seam_val 沿用 prereg02（= prereg01 推論時的數字）
EXPECTED_COUNTS = {'seam_val': (P01_SUMMARY['n_candidate_rows'], P01_SUMMARY['n_query_games'], P01_SUMMARY['n_questions'])}
for _pool in ('public', 'private'):
    _qi, _ci = QUICK[_pool]['q'], QUICK[_pool]['c']
    EXPECTED_COUNTS[_pool] = (smoke_counts(_qi, _ci) if SMOKE else
                              (_ci['rows'], sum(_qi['num_games']), _qi['rows']))
POOL_ORDER = ('public', 'private', 'seam_val')      # seam_val 最後：dump 格開頭會刪掉上一次的重現核對結果
POOL_FILES = {'public': tuple(os.path.join(TESTS_DIR, n) for n in TEST_POOLS['public']),
              'private': tuple(os.path.join(TESTS_DIR, n) for n in TEST_POOLS['private']),
              'seam_val': (QUESTIONS_CSV, CANDIDATES_CSV)}
DUMP_INFO_BY_POOL = {}


# ---- test02：stale-dump guard（上一次用不同輸入檔算的 dump 還在就停下）
def input_record(pool):
    _q, _c = POOL_FILES[pool]
    return {'set': pool, 'question_file': os.path.basename(_q), 'sha256': sha256_file(_q),
            'candidates_file': os.path.basename(_c), 'candidates_sha256': sha256_file(_c)}


INPUT_RECORDS = {_pool: input_record(_pool) for _pool in POOL_ORDER}
_stale = []
for _pool, _rec in INPUT_RECORDS.items():
    _p = os.path.join(RESULT_DIR, _pool, 'dumps', '_INPUT.json')
    if os.path.exists(_p):
        with open(_p, encoding='utf-8') as f:
            _old = json.load(f)
        if _old != _rec:
            _stale.append('%s：上一次的題目檔 %s…、候選檔 %s…；這一次 %s…、%s…'
                          % (_pool, str(_old.get('sha256'))[:12], str(_old.get('candidates_sha256'))[:12],
                             _rec['sha256'][:12], _rec['candidates_sha256'][:12]))
if _stale:
    preflight_stop('%s 裡有上一次用不同輸入檔算的 dump：\n%s\n處理方式：不要自己刪檔，截圖給 Claude。'
                   % (RESULT_DIR, '\n'.join('  - ' + x for x in _stale)))


def write_input_json(pool):
    # dump 格呼叫：記下這個池用的題目檔與候選檔（下一次執行的 stale-dump guard 用）
    _p = guard_write(os.path.join(RESULT_DIR, pool, 'dumps', '_INPUT.json'))
    with open(_p + '.tmp', 'w', encoding='utf-8') as f:
        json.dump(INPUT_RECORDS[pool], f, ensure_ascii=False, indent=1)
    os.replace(_p + '.tmp', _p)


CONFIG['p01_inputs'] = {
    'root': P01_ROOT,
    'files': {k: {'path': p, 'sha256': P01_SHA256[k], 'bytes': os.path.getsize(p)} for k, p in P01_FILES.items()},
    'checked_against_repo': not SMOKE,
    'predict_summary': P01_SUMMARY,
    'p01_gpu': P01_ENV.get('gpu') if P01_ENV else None,
    'p01_torch': P01_ENV.get('torch') if P01_ENV else None,
    'questions_csv_sha256': QUESTIONS_SHA256,
    'candidates_csv_sha256': CANDIDATES_SHA256,
}
CONFIG['weights'] = {'path': os.path.relpath(P01_FILES['best_model.pth'], os.path.join(DRIVE_DIR, 'results')).replace(os.sep, '/'),
                     'sha256': P01_SHA256['best_model.pth']}
CONFIG['p02_inputs'] = {
    'root': P02_ROOT,
    'dumps': {k: {'path': p, 'sha256': P02_SHA256[k]} for k, p in P02_DUMPS.items()},
    'checked_against_repo': not SMOKE,
    'p02_gpu': P02_ENV.get('gpu') if P02_ENV else None,
    'p02_torch': P02_ENV.get('torch') if P02_ENV else None,
    'p02_cudnn': P02_ENV.get('cudnn') if P02_ENV else None,
}
CONFIG['test_bundle'] = {
    'zip': TEST_ZIP, 'zip_sha256': TEST_ZIP_SHA256, 'manifest_sha256': TEST_MANIFEST_SHA256,
    'kind': TEST_MANIFEST.get('kind'), 'created': TEST_MANIFEST.get('created'), 't1': TEST_MANIFEST.get('t1'),
    'files': TEST_SHA256,
    'names_checked_only': [n for n in TEST_NAMES if n not in PLAYER_NAMES],
}
_seam_rows = len(pd.read_csv(QUESTIONS_CSV, usecols=['question_id']))
CONFIG['sets'] = {_pool: {'questions_file': _rec['question_file'], 'questions_sha256': _rec['sha256'],
                          'candidates_file': _rec['candidates_file'], 'candidates_sha256': _rec['candidates_sha256'],
                          'rows': (QUICK[_pool]['q']['rows'] if _pool in QUICK else _seam_rows),
                          'expected_dump_counts': dict(zip(('candidate_games', 'query_games', 'questions'),
                                                           EXPECTED_COUNTS[_pool]))}
                  for _pool, _rec in INPUT_RECORDS.items()}
save_json('config.json', CONFIG)
update_run(p01_sha256=P01_SHA256, p02_dump_sha256=P02_SHA256, test_zip_sha256=TEST_ZIP_SHA256, drive_free_gb=_free_gb)
record_time('preflight_s', time.time() - _t_pre)
print('preflight 通過：prereg01 的 4 個檔都在，SHA-256 相符%s；Drive 剩 %s'
      % ('（含 repo 記錄）' if not SMOKE else '（SMOKE 不比 repo 記錄）',
         '%.1f GB' % _free_gb if _free_gb is not None else '（查不到）'))
for _k in P01_FILES:
    print('  %s  %s' % (P01_SHA256[_k][:16], _k))
for _k in P02_DUMPS:
    print('  %s  prereg02 dumps/%s' % (P02_SHA256[_k][:16], _k))
print('測試包 %s（kind %s）：競賽一四個檔 SHA-256 和 MANIFEST 相同，快速格式檢查通過' % (TEST_ZIP_SHA256[:16], TEST_MANIFEST.get('kind')))
for _pool in POOL_ORDER:
    print('  %-8s 候選 %d 盤、題目 %d 盤、%d 題（dump 格照這個核對）' % ((_pool,) + tuple(EXPECTED_COUNTS[_pool])))


In [ ]:
GPU_BATCH = 4096
CHUNK = 256
# prereg02：只留 sgf_windows、embed（官方原文）；每盤的 embedding 由下一格的 game_embeddings() 存
_infer_parser = SGFParsePlayerIdentification()


def sgf_windows(args):
    """給 multiprocessing.Pool 用的頂層函式。(sgf, color) → (N, 17, 19, 19)"""
    sgf, color = args
    try:
        feats = _infer_parser.extract_features(sgf, str(color).lower(),
                                               mode='inference')
        return np.asarray(feats, dtype=np.uint8) if len(feats) else None
    except Exception:
        return None


def embed(model, arr):
    outs = []
    with torch.no_grad():
        for i in range(0, len(arr), GPU_BATCH):
            x = torch.from_numpy(arr[i:i + GPU_BATCH]).float().to(DEVICE)
            outs.append(model(x).cpu().numpy())
    return np.concatenate(outs) if outs else np.zeros((0, 256), np.float32)


### 每盤 embedding：public、private、seam_val 各一格（登記第 9 節第 4 條）

三格的程式完全相同，只差第一行的 `POOL`。每盤的算法和 prereg02 相同（`sgf_windows` → 一盤一次 `embed` → 窗口 embedding 在 float32 下加總；`game_embeddings()` 和候選、題目兩段迴圈逐位元組相同）。輸出寫到 `results/test02/player/<POOL>/dumps/`，另寫 `_INPUT.json`。盤數和題數照 preflight 算好的數字核對。seam_val（本機驗證池）放最後：這一格開頭會刪掉上一次的重現核對結果，重現核對格接著用 seam_val 的結果。

In [ ]:
POOL = 'public'; QUESTIONS_CSV, CANDIDATES_CSV = POOL_FILES[POOL]  # test02：這一格處理的候選池
# 每盤 embedding（test02，登記第 9 節第 4 條）：POOL 這個池的候選 → <POOL>/dumps/cand_games.npz；題目 → <POOL>/dumps/query_games.npz
import io

_t_dump = time.time()
DUMP_DIR = os.path.join(RESULT_DIR, POOL, 'dumps')
CAND_NPZ = os.path.join(DUMP_DIR, 'cand_games.npz')
QUERY_NPZ = os.path.join(DUMP_DIR, 'query_games.npz')
REPRO_JSON = os.path.join(RESULT_DIR, 'repro_check.json')
REPRO_CSV = os.path.join(RESULT_DIR, 'player_val_top5_repro.csv')
EMBED_DIM = 256

# 沒有接續：整份重跑。先刪掉上一次留下的 dump、重現核對結果與結尾指紋，避免新舊混用
for _p in (CAND_NPZ, QUERY_NPZ, CAND_NPZ + '.tmp', QUERY_NPZ + '.tmp', REPRO_JSON, REPRO_CSV,
           os.path.join(RESULT_DIR, 'readonly_after.json'), os.path.join(RESULT_DIR, 'seam_p02.json'),
           os.path.join(DUMP_DIR, '_INPUT.json')):
    if os.path.exists(_p):
        os.remove(guard_write(_p))
os.makedirs(guard_write(DUMP_DIR), exist_ok=True)
write_input_json(POOL)          # test02：這個池用的題目檔與候選檔（檔名與 SHA-256）
DUMP_INFO_BY_POOL.pop(POOL, None)


def game_embeddings(model, pool, jobs, label):
    # jobs：[(sgf, color)]。每盤一列：窗口 embedding 的加總（float32）與窗口數。
    # 每盤的算法和 prereg01 的 mean_embeddings 相同（每次平行解析 CHUNK 盤；一盤一次 embed；e.sum(0) 是 float32），
    # 只是不按 key 加總。sgf_windows 回傳 None（解析失敗或沒有窗口）的盤：加總全 0、窗口數 0，prereg01 也跳過這種盤
    emb_sum = np.zeros((len(jobs), EMBED_DIM), np.float32)
    n_windows = np.zeros(len(jobs), np.int32)
    t0 = time.time()
    for start in range(0, len(jobs), CHUNK):
        for g, arr in zip(range(start, start + CHUNK), pool.map(sgf_windows, jobs[start:start + CHUNK])):
            if arr is None:
                continue
            e = embed(model, arr)
            assert e.dtype == np.float32 and e.shape == (len(arr), EMBED_DIM), (e.dtype, e.shape)
            emb_sum[g] = e.sum(0)
            n_windows[g] = len(e)
        done = min(start + CHUNK, len(jobs))
        if done % (CHUNK * 20) == 0 or done == len(jobs):
            print('  %s %d/%d 局  %.0f 秒' % (label, done, len(jobs), time.time() - t0), flush=True)
    return emb_sum, n_windows


def save_npz(path, **arrays):
    # 先寫暫存檔再改名；np.savez 收到檔名會自己補 .npz，所以用檔案物件寫
    with open(guard_write(path + '.tmp'), 'wb') as f:
        np.savez(f, **arrays)
    os.replace(path + '.tmp', guard_write(path))
    return {'rows': int(len(arrays['n_windows'])), 'sha256': sha256_file(path), 'bytes': os.path.getsize(path),
            'n_parsed': int((arrays['n_windows'] > 0).sum()), 'n_windows_total': int(arrays['n_windows'].sum())}


# 權重：整檔讀進記憶體，SHA-256 和 preflight 記的相同才載入（載入的就是算過 SHA-256 的那份位元組）
with open(P01_FILES['best_model.pth'], 'rb') as f:
    _wbytes = f.read()
WEIGHTS_SHA256 = hashlib.sha256(_wbytes).hexdigest()
if WEIGHTS_SHA256 != P01_SHA256['best_model.pth']:
    raise SystemExit('best_model.pth 在 preflight 之後變了（SHA-256 %s），停止' % WEIGHTS_SHA256)
net = GoPlayerResNet()
net.load_state_dict(torch.load(io.BytesIO(_wbytes), map_location='cpu'))
net.eval().to(DEVICE)
del _wbytes

DUMP_INFO = {'weights_sha256': WEIGHTS_SHA256, 'GPU_BATCH': GPU_BATCH, 'CHUNK': CHUNK, 'NPROC': NPROC,
             'device': str(DEVICE), 'gpu': ENV['gpu']}
with Pool(NPROC) as pool:
    # 候選：和 prereg01 同一個檔、同一個列順序；SMOKE 的縮小方式也和 prereg01 相同
    CAND = pd.read_csv(CANDIDATES_CSV)
    if SMOKE:
        _keep = sorted(CAND['player_id'].unique())[:SMOKE_N_CAND_PLAYERS]
        CAND = (CAND[CAND['player_id'].isin(_keep)].groupby('player_id', sort=False)
                .head(SMOKE_CAND_GAMES).reset_index(drop=True))
    _t = time.time()
    CAND_SUM, CAND_NW = game_embeddings(net, pool, list(zip(CAND['sgf_content'], CAND['color'])), '候選')
    record_time('embed_candidates_s', time.time() - _t)
    DUMP_INFO['cand_games.npz'] = save_npz(
        CAND_NPZ, game_id=np.array(CAND['game_id'].astype(str).tolist()),
        player_id=np.array(CAND['player_id'].astype(str).tolist()),
        color=np.array(CAND['color'].astype(str).tolist()),
        emb_sum=CAND_SUM, n_windows=CAND_NW, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (CAND_NPZ, len(CAND)))

    # 題目：題目檔的列順序；每題依 sgf_1、sgf_2… 的順序，只收非空的欄（同 prereg01）；slot 是欄位編號
    TEST = pd.read_csv(QUESTIONS_CSV)
    if SMOKE:
        TEST = TEST.head(SMOKE_N_QUESTIONS).reset_index(drop=True)
    sgf_cols = [c for c in TEST.columns if c.startswith('sgf_')]
    col_cols = [c for c in TEST.columns if c.startswith('color_')]
    jobs, Q_IDS, Q_SLOTS, Q_COLORS = [], [], [], []
    for _, r in TEST.iterrows():
        used = 0
        # num_games 之後的 sgf_ 欄位是空字串，要濾掉
        for sc, cc in zip(sgf_cols, col_cols):
            if SMOKE and used >= SMOKE_Q_GAMES:
                break
            if pd.notna(r[sc]) and str(r[sc]).strip():
                _slot = int(sc[len('sgf_'):])
                assert cc == 'color_%d' % _slot, (sc, cc)
                jobs.append((r[sc], r[cc]))
                Q_IDS.append(str(r['question_id']))
                Q_SLOTS.append(_slot)
                Q_COLORS.append(str(r[cc]))
                used += 1
    _t = time.time()
    Q_SUM, Q_NW = game_embeddings(net, pool, jobs, '查詢')
    record_time('embed_queries_s', time.time() - _t)
    DUMP_INFO['query_games.npz'] = save_npz(
        QUERY_NPZ, question_id=np.array(Q_IDS), slot=np.array(Q_SLOTS, np.int32), color=np.array(Q_COLORS),
        emb_sum=Q_SUM, n_windows=Q_NW, parsed=Q_NW > 0, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (QUERY_NPZ, len(jobs)))

N_CAND_ROWS, N_QUERY_ROWS = len(CAND), len(jobs)
# 盤數和題數必須和 preflight 算好的相同（seam_val：和 prereg01 推論時相同；SMOKE 時照同樣的縮小方式）
assert (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)) == tuple(EXPECTED_COUNTS[POOL]), \
    (POOL, (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)), EXPECTED_COUNTS[POOL])
DUMP_INFO['seconds'] = time.time() - _t_dump
DUMP_INFO_BY_POOL[POOL] = DUMP_INFO
update_run(dumps=DUMP_INFO_BY_POOL)
record_time(POOL + '_dump_total_s', DUMP_INFO['seconds'])
print('%s 每盤 embedding 完成：候選 %d 盤（可解析 %d）、題目 %d 盤（可解析 %d），%.0f 秒'
      % (POOL, N_CAND_ROWS, DUMP_INFO['cand_games.npz']['n_parsed'], N_QUERY_ROWS,
         DUMP_INFO['query_games.npz']['n_parsed'], DUMP_INFO['seconds']))


In [ ]:
POOL = 'private'; QUESTIONS_CSV, CANDIDATES_CSV = POOL_FILES[POOL]  # test02：這一格處理的候選池
# 每盤 embedding（test02，登記第 9 節第 4 條）：POOL 這個池的候選 → <POOL>/dumps/cand_games.npz；題目 → <POOL>/dumps/query_games.npz
import io

_t_dump = time.time()
DUMP_DIR = os.path.join(RESULT_DIR, POOL, 'dumps')
CAND_NPZ = os.path.join(DUMP_DIR, 'cand_games.npz')
QUERY_NPZ = os.path.join(DUMP_DIR, 'query_games.npz')
REPRO_JSON = os.path.join(RESULT_DIR, 'repro_check.json')
REPRO_CSV = os.path.join(RESULT_DIR, 'player_val_top5_repro.csv')
EMBED_DIM = 256

# 沒有接續：整份重跑。先刪掉上一次留下的 dump、重現核對結果與結尾指紋，避免新舊混用
for _p in (CAND_NPZ, QUERY_NPZ, CAND_NPZ + '.tmp', QUERY_NPZ + '.tmp', REPRO_JSON, REPRO_CSV,
           os.path.join(RESULT_DIR, 'readonly_after.json'), os.path.join(RESULT_DIR, 'seam_p02.json'),
           os.path.join(DUMP_DIR, '_INPUT.json')):
    if os.path.exists(_p):
        os.remove(guard_write(_p))
os.makedirs(guard_write(DUMP_DIR), exist_ok=True)
write_input_json(POOL)          # test02：這個池用的題目檔與候選檔（檔名與 SHA-256）
DUMP_INFO_BY_POOL.pop(POOL, None)


def game_embeddings(model, pool, jobs, label):
    # jobs：[(sgf, color)]。每盤一列：窗口 embedding 的加總（float32）與窗口數。
    # 每盤的算法和 prereg01 的 mean_embeddings 相同（每次平行解析 CHUNK 盤；一盤一次 embed；e.sum(0) 是 float32），
    # 只是不按 key 加總。sgf_windows 回傳 None（解析失敗或沒有窗口）的盤：加總全 0、窗口數 0，prereg01 也跳過這種盤
    emb_sum = np.zeros((len(jobs), EMBED_DIM), np.float32)
    n_windows = np.zeros(len(jobs), np.int32)
    t0 = time.time()
    for start in range(0, len(jobs), CHUNK):
        for g, arr in zip(range(start, start + CHUNK), pool.map(sgf_windows, jobs[start:start + CHUNK])):
            if arr is None:
                continue
            e = embed(model, arr)
            assert e.dtype == np.float32 and e.shape == (len(arr), EMBED_DIM), (e.dtype, e.shape)
            emb_sum[g] = e.sum(0)
            n_windows[g] = len(e)
        done = min(start + CHUNK, len(jobs))
        if done % (CHUNK * 20) == 0 or done == len(jobs):
            print('  %s %d/%d 局  %.0f 秒' % (label, done, len(jobs), time.time() - t0), flush=True)
    return emb_sum, n_windows


def save_npz(path, **arrays):
    # 先寫暫存檔再改名；np.savez 收到檔名會自己補 .npz，所以用檔案物件寫
    with open(guard_write(path + '.tmp'), 'wb') as f:
        np.savez(f, **arrays)
    os.replace(path + '.tmp', guard_write(path))
    return {'rows': int(len(arrays['n_windows'])), 'sha256': sha256_file(path), 'bytes': os.path.getsize(path),
            'n_parsed': int((arrays['n_windows'] > 0).sum()), 'n_windows_total': int(arrays['n_windows'].sum())}


# 權重：整檔讀進記憶體，SHA-256 和 preflight 記的相同才載入（載入的就是算過 SHA-256 的那份位元組）
with open(P01_FILES['best_model.pth'], 'rb') as f:
    _wbytes = f.read()
WEIGHTS_SHA256 = hashlib.sha256(_wbytes).hexdigest()
if WEIGHTS_SHA256 != P01_SHA256['best_model.pth']:
    raise SystemExit('best_model.pth 在 preflight 之後變了（SHA-256 %s），停止' % WEIGHTS_SHA256)
net = GoPlayerResNet()
net.load_state_dict(torch.load(io.BytesIO(_wbytes), map_location='cpu'))
net.eval().to(DEVICE)
del _wbytes

DUMP_INFO = {'weights_sha256': WEIGHTS_SHA256, 'GPU_BATCH': GPU_BATCH, 'CHUNK': CHUNK, 'NPROC': NPROC,
             'device': str(DEVICE), 'gpu': ENV['gpu']}
with Pool(NPROC) as pool:
    # 候選：和 prereg01 同一個檔、同一個列順序；SMOKE 的縮小方式也和 prereg01 相同
    CAND = pd.read_csv(CANDIDATES_CSV)
    if SMOKE:
        _keep = sorted(CAND['player_id'].unique())[:SMOKE_N_CAND_PLAYERS]
        CAND = (CAND[CAND['player_id'].isin(_keep)].groupby('player_id', sort=False)
                .head(SMOKE_CAND_GAMES).reset_index(drop=True))
    _t = time.time()
    CAND_SUM, CAND_NW = game_embeddings(net, pool, list(zip(CAND['sgf_content'], CAND['color'])), '候選')
    record_time('embed_candidates_s', time.time() - _t)
    DUMP_INFO['cand_games.npz'] = save_npz(
        CAND_NPZ, game_id=np.array(CAND['game_id'].astype(str).tolist()),
        player_id=np.array(CAND['player_id'].astype(str).tolist()),
        color=np.array(CAND['color'].astype(str).tolist()),
        emb_sum=CAND_SUM, n_windows=CAND_NW, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (CAND_NPZ, len(CAND)))

    # 題目：題目檔的列順序；每題依 sgf_1、sgf_2… 的順序，只收非空的欄（同 prereg01）；slot 是欄位編號
    TEST = pd.read_csv(QUESTIONS_CSV)
    if SMOKE:
        TEST = TEST.head(SMOKE_N_QUESTIONS).reset_index(drop=True)
    sgf_cols = [c for c in TEST.columns if c.startswith('sgf_')]
    col_cols = [c for c in TEST.columns if c.startswith('color_')]
    jobs, Q_IDS, Q_SLOTS, Q_COLORS = [], [], [], []
    for _, r in TEST.iterrows():
        used = 0
        # num_games 之後的 sgf_ 欄位是空字串，要濾掉
        for sc, cc in zip(sgf_cols, col_cols):
            if SMOKE and used >= SMOKE_Q_GAMES:
                break
            if pd.notna(r[sc]) and str(r[sc]).strip():
                _slot = int(sc[len('sgf_'):])
                assert cc == 'color_%d' % _slot, (sc, cc)
                jobs.append((r[sc], r[cc]))
                Q_IDS.append(str(r['question_id']))
                Q_SLOTS.append(_slot)
                Q_COLORS.append(str(r[cc]))
                used += 1
    _t = time.time()
    Q_SUM, Q_NW = game_embeddings(net, pool, jobs, '查詢')
    record_time('embed_queries_s', time.time() - _t)
    DUMP_INFO['query_games.npz'] = save_npz(
        QUERY_NPZ, question_id=np.array(Q_IDS), slot=np.array(Q_SLOTS, np.int32), color=np.array(Q_COLORS),
        emb_sum=Q_SUM, n_windows=Q_NW, parsed=Q_NW > 0, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (QUERY_NPZ, len(jobs)))

N_CAND_ROWS, N_QUERY_ROWS = len(CAND), len(jobs)
# 盤數和題數必須和 preflight 算好的相同（seam_val：和 prereg01 推論時相同；SMOKE 時照同樣的縮小方式）
assert (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)) == tuple(EXPECTED_COUNTS[POOL]), \
    (POOL, (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)), EXPECTED_COUNTS[POOL])
DUMP_INFO['seconds'] = time.time() - _t_dump
DUMP_INFO_BY_POOL[POOL] = DUMP_INFO
update_run(dumps=DUMP_INFO_BY_POOL)
record_time(POOL + '_dump_total_s', DUMP_INFO['seconds'])
print('%s 每盤 embedding 完成：候選 %d 盤（可解析 %d）、題目 %d 盤（可解析 %d），%.0f 秒'
      % (POOL, N_CAND_ROWS, DUMP_INFO['cand_games.npz']['n_parsed'], N_QUERY_ROWS,
         DUMP_INFO['query_games.npz']['n_parsed'], DUMP_INFO['seconds']))


In [ ]:
POOL = 'seam_val'; QUESTIONS_CSV, CANDIDATES_CSV = POOL_FILES[POOL]  # test02：這一格處理的候選池
# 每盤 embedding（test02，登記第 9 節第 4 條）：POOL 這個池的候選 → <POOL>/dumps/cand_games.npz；題目 → <POOL>/dumps/query_games.npz
import io

_t_dump = time.time()
DUMP_DIR = os.path.join(RESULT_DIR, POOL, 'dumps')
CAND_NPZ = os.path.join(DUMP_DIR, 'cand_games.npz')
QUERY_NPZ = os.path.join(DUMP_DIR, 'query_games.npz')
REPRO_JSON = os.path.join(RESULT_DIR, 'repro_check.json')
REPRO_CSV = os.path.join(RESULT_DIR, 'player_val_top5_repro.csv')
EMBED_DIM = 256

# 沒有接續：整份重跑。先刪掉上一次留下的 dump、重現核對結果與結尾指紋，避免新舊混用
for _p in (CAND_NPZ, QUERY_NPZ, CAND_NPZ + '.tmp', QUERY_NPZ + '.tmp', REPRO_JSON, REPRO_CSV,
           os.path.join(RESULT_DIR, 'readonly_after.json'), os.path.join(RESULT_DIR, 'seam_p02.json'),
           os.path.join(DUMP_DIR, '_INPUT.json')):
    if os.path.exists(_p):
        os.remove(guard_write(_p))
os.makedirs(guard_write(DUMP_DIR), exist_ok=True)
write_input_json(POOL)          # test02：這個池用的題目檔與候選檔（檔名與 SHA-256）
DUMP_INFO_BY_POOL.pop(POOL, None)


def game_embeddings(model, pool, jobs, label):
    # jobs：[(sgf, color)]。每盤一列：窗口 embedding 的加總（float32）與窗口數。
    # 每盤的算法和 prereg01 的 mean_embeddings 相同（每次平行解析 CHUNK 盤；一盤一次 embed；e.sum(0) 是 float32），
    # 只是不按 key 加總。sgf_windows 回傳 None（解析失敗或沒有窗口）的盤：加總全 0、窗口數 0，prereg01 也跳過這種盤
    emb_sum = np.zeros((len(jobs), EMBED_DIM), np.float32)
    n_windows = np.zeros(len(jobs), np.int32)
    t0 = time.time()
    for start in range(0, len(jobs), CHUNK):
        for g, arr in zip(range(start, start + CHUNK), pool.map(sgf_windows, jobs[start:start + CHUNK])):
            if arr is None:
                continue
            e = embed(model, arr)
            assert e.dtype == np.float32 and e.shape == (len(arr), EMBED_DIM), (e.dtype, e.shape)
            emb_sum[g] = e.sum(0)
            n_windows[g] = len(e)
        done = min(start + CHUNK, len(jobs))
        if done % (CHUNK * 20) == 0 or done == len(jobs):
            print('  %s %d/%d 局  %.0f 秒' % (label, done, len(jobs), time.time() - t0), flush=True)
    return emb_sum, n_windows


def save_npz(path, **arrays):
    # 先寫暫存檔再改名；np.savez 收到檔名會自己補 .npz，所以用檔案物件寫
    with open(guard_write(path + '.tmp'), 'wb') as f:
        np.savez(f, **arrays)
    os.replace(path + '.tmp', guard_write(path))
    return {'rows': int(len(arrays['n_windows'])), 'sha256': sha256_file(path), 'bytes': os.path.getsize(path),
            'n_parsed': int((arrays['n_windows'] > 0).sum()), 'n_windows_total': int(arrays['n_windows'].sum())}


# 權重：整檔讀進記憶體，SHA-256 和 preflight 記的相同才載入（載入的就是算過 SHA-256 的那份位元組）
with open(P01_FILES['best_model.pth'], 'rb') as f:
    _wbytes = f.read()
WEIGHTS_SHA256 = hashlib.sha256(_wbytes).hexdigest()
if WEIGHTS_SHA256 != P01_SHA256['best_model.pth']:
    raise SystemExit('best_model.pth 在 preflight 之後變了（SHA-256 %s），停止' % WEIGHTS_SHA256)
net = GoPlayerResNet()
net.load_state_dict(torch.load(io.BytesIO(_wbytes), map_location='cpu'))
net.eval().to(DEVICE)
del _wbytes

DUMP_INFO = {'weights_sha256': WEIGHTS_SHA256, 'GPU_BATCH': GPU_BATCH, 'CHUNK': CHUNK, 'NPROC': NPROC,
             'device': str(DEVICE), 'gpu': ENV['gpu']}
with Pool(NPROC) as pool:
    # 候選：和 prereg01 同一個檔、同一個列順序；SMOKE 的縮小方式也和 prereg01 相同
    CAND = pd.read_csv(CANDIDATES_CSV)
    if SMOKE:
        _keep = sorted(CAND['player_id'].unique())[:SMOKE_N_CAND_PLAYERS]
        CAND = (CAND[CAND['player_id'].isin(_keep)].groupby('player_id', sort=False)
                .head(SMOKE_CAND_GAMES).reset_index(drop=True))
    _t = time.time()
    CAND_SUM, CAND_NW = game_embeddings(net, pool, list(zip(CAND['sgf_content'], CAND['color'])), '候選')
    record_time('embed_candidates_s', time.time() - _t)
    DUMP_INFO['cand_games.npz'] = save_npz(
        CAND_NPZ, game_id=np.array(CAND['game_id'].astype(str).tolist()),
        player_id=np.array(CAND['player_id'].astype(str).tolist()),
        color=np.array(CAND['color'].astype(str).tolist()),
        emb_sum=CAND_SUM, n_windows=CAND_NW, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (CAND_NPZ, len(CAND)))

    # 題目：題目檔的列順序；每題依 sgf_1、sgf_2… 的順序，只收非空的欄（同 prereg01）；slot 是欄位編號
    TEST = pd.read_csv(QUESTIONS_CSV)
    if SMOKE:
        TEST = TEST.head(SMOKE_N_QUESTIONS).reset_index(drop=True)
    sgf_cols = [c for c in TEST.columns if c.startswith('sgf_')]
    col_cols = [c for c in TEST.columns if c.startswith('color_')]
    jobs, Q_IDS, Q_SLOTS, Q_COLORS = [], [], [], []
    for _, r in TEST.iterrows():
        used = 0
        # num_games 之後的 sgf_ 欄位是空字串，要濾掉
        for sc, cc in zip(sgf_cols, col_cols):
            if SMOKE and used >= SMOKE_Q_GAMES:
                break
            if pd.notna(r[sc]) and str(r[sc]).strip():
                _slot = int(sc[len('sgf_'):])
                assert cc == 'color_%d' % _slot, (sc, cc)
                jobs.append((r[sc], r[cc]))
                Q_IDS.append(str(r['question_id']))
                Q_SLOTS.append(_slot)
                Q_COLORS.append(str(r[cc]))
                used += 1
    _t = time.time()
    Q_SUM, Q_NW = game_embeddings(net, pool, jobs, '查詢')
    record_time('embed_queries_s', time.time() - _t)
    DUMP_INFO['query_games.npz'] = save_npz(
        QUERY_NPZ, question_id=np.array(Q_IDS), slot=np.array(Q_SLOTS, np.int32), color=np.array(Q_COLORS),
        emb_sum=Q_SUM, n_windows=Q_NW, parsed=Q_NW > 0, weights_sha256=np.array(WEIGHTS_SHA256))
    print('寫出 %s（%d 列）' % (QUERY_NPZ, len(jobs)))

N_CAND_ROWS, N_QUERY_ROWS = len(CAND), len(jobs)
# 盤數和題數必須和 preflight 算好的相同（seam_val：和 prereg01 推論時相同；SMOKE 時照同樣的縮小方式）
assert (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)) == tuple(EXPECTED_COUNTS[POOL]), \
    (POOL, (N_CAND_ROWS, N_QUERY_ROWS, len(TEST)), EXPECTED_COUNTS[POOL])
DUMP_INFO['seconds'] = time.time() - _t_dump
DUMP_INFO_BY_POOL[POOL] = DUMP_INFO
update_run(dumps=DUMP_INFO_BY_POOL)
record_time(POOL + '_dump_total_s', DUMP_INFO['seconds'])
print('%s 每盤 embedding 完成：候選 %d 盤（可解析 %d）、題目 %d 盤（可解析 %d），%.0f 秒'
      % (POOL, N_CAND_ROWS, DUMP_INFO['cand_games.npz']['n_parsed'], N_QUERY_ROWS,
         DUMP_INFO['query_games.npz']['n_parsed'], DUMP_INFO['seconds']))


### 本版新增：重現核對（登記 2.1）

只用上一格存的每盤 embedding，照 prereg01 的順序與精度重算依窗口數加權的重心和題目向量，照官方做法用 L2 距離取 Top-5（`player_val_top5_repro.csv`），再和 prereg01 比。通過條件：400 題中至少 398 題的 Top-5 完全相同（SMOKE：全部相同），候選代號與順序相同，重心最大差距 ≤ 1e-4。結果寫 `repro_check.json`。沒過的話收尾格照樣打包，最後停下。

In [ ]:
# 重現核對（本版新增，登記 2.1）：用每盤 embedding 重算 prereg01 的重心與 Top-5（E0），和 prereg01 的輸出比
_t_repro = time.time()


def weighted_means(keys, emb_sum, n_windows):
    # 和 prereg01 mean_embeddings 相同的順序與精度：依列順序逐盤在 float32 下累加，再除以窗口數的和（Python int）
    sums, counts = {}, {}
    for k, e, n in zip(keys, emb_sum, n_windows):
        if n == 0:
            continue          # prereg01 也跳過 sgf_windows 回傳 None 的盤
        sums[k] = sums.get(k, 0) + e
        counts[k] = counts.get(k, 0) + int(n)
    return {k: sums[k] / counts[k] for k in sums}


centroids = weighted_means(CAND['player_id'].astype(str).tolist(), CAND_SUM, CAND_NW)
names = sorted(centroids)          # 候選清單照玩家代號排序（同官方）
C = np.stack([centroids[n] for n in names])
queries = weighted_means(Q_IDS, Q_SUM, Q_NW)
rows, n_fallback = [], 0
for qid in TEST['question_id'].astype(str):
    q = queries.get(qid)
    if q is None:
        rows.append([qid] + names[:TOP_N])
        n_fallback += 1
        continue
    d = np.linalg.norm(C - q, axis=1)
    rows.append([qid] + [names[i] for i in np.argsort(d)[:TOP_N]])
_cols = ['question_id'] + ['top_%d' % i for i in range(1, TOP_N + 1)]
REPRO_TOP5 = pd.DataFrame(rows, columns=_cols)
REPRO_TOP5.to_csv(guard_write(REPRO_CSV + '.tmp'), index=False)
os.replace(REPRO_CSV + '.tmp', guard_write(REPRO_CSV))

# 和 prereg01 的 Top-5 逐題比（top_1…top_5 全部相同、順序也相同才算相同）
P01_TOP5 = pd.read_csv(P01_FILES['player_val_top5.csv'], dtype=str, keep_default_na=False)
_same_qids = P01_TOP5['question_id'].tolist() == REPRO_TOP5['question_id'].tolist()
_mismatch = []
if _same_qids:
    for _a, _b in zip(P01_TOP5[_cols].values.tolist(), REPRO_TOP5[_cols].values.tolist()):
        if _a != _b:
            _mismatch.append({'question_id': _a[0], 'prereg01': _a[1:], 'repro': _b[1:]})

# 和 prereg01 的重心、題目向量比（候選代號與順序必須相同；題目向量的差距只記錄）
with np.load(P01_FILES['player_val_embeddings.npz']) as _z:
    _p01_cids = [str(x) for x in _z['candidate_ids']]
    _p01_C = _z['centroids']
    _p01_qids = [str(x) for x in _z['question_ids']]
    _p01_Q = _z['queries']
_same_cids = _p01_cids == names and _p01_C.shape == C.shape
_centroid_diff = (float(np.max(np.abs(C.astype(np.float64) - _p01_C.astype(np.float64)))) if _same_cids and len(C)
                  else None)
_q_ids_emb = [q for q in TEST['question_id'].astype(str) if q in queries]
Q = np.stack([queries[q] for q in _q_ids_emb]) if _q_ids_emb else np.zeros((0, EMBED_DIM), np.float32)
_same_qids_emb = _p01_qids == _q_ids_emb and _p01_Q.shape == Q.shape
_query_diff = (float(np.max(np.abs(Q.astype(np.float64) - _p01_Q.astype(np.float64)))) if _same_qids_emb and len(Q)
               else None)

_n_q = len(REPRO_TOP5)
_n_identical = _n_q - len(_mismatch) if _same_qids else 0
_criteria = {
    'question_ids_same_order': _same_qids,
    'questions_count_ok': SMOKE or _n_q == 400,
    'top5_identical_ok': _same_qids and len(_mismatch) <= REPRO_MAX_MISMATCH,
    'candidate_ids_same_order': _same_cids,
    'centroid_diff_ok': _centroid_diff is not None and _centroid_diff <= REPRO_CENTROID_TOL,
}
_p01_gpu = CONFIG['p01_inputs']['p01_gpu']
REPRO = {
    'checked_utc': now_utc(),
    'smoke': SMOKE,
    'method': 'E0：依窗口數加權的重心、L2 距離、np.argsort 取前 %d（同 prereg01）；只用 dumps/ 的每盤 embedding 重算' % TOP_N,
    'n_questions': _n_q,
    'n_top5_identical': _n_identical,
    'n_top5_mismatch': len(_mismatch) if _same_qids else None,
    'top5_max_mismatch_allowed': REPRO_MAX_MISMATCH,
    'mismatched_questions': _mismatch,
    'n_fallback_questions': n_fallback,
    'n_centroids': len(names),
    'centroid_max_abs_diff': _centroid_diff,
    'centroid_tol': REPRO_CENTROID_TOL,
    'centroids_bitwise_identical': bool(_same_cids and np.array_equal(C, _p01_C)),
    'query_max_abs_diff_info_only': _query_diff,
    'query_ids_same_order_info_only': _same_qids_emb,
    'gpu': ENV['gpu'],
    'p01_gpu': _p01_gpu,
    'same_gpu_as_p01': (ENV['gpu'] == _p01_gpu) if _p01_gpu is not None else None,
    'torch': ENV['torch'],
    'p01_torch': CONFIG['p01_inputs']['p01_torch'],
    'weights_sha256': WEIGHTS_SHA256,
    'p01_top5_sha256': P01_SHA256['player_val_top5.csv'],
    'p01_embeddings_sha256': P01_SHA256['player_val_embeddings.npz'],
    'repro_top5_sha256': sha256_file(REPRO_CSV),
    'dumps': {k: DUMP_INFO[k] for k in ('cand_games.npz', 'query_games.npz')},
    'criteria': _criteria,
    'passed': all(_criteria.values()),
}
save_json('repro_check.json', REPRO)
update_run(repro_passed=REPRO['passed'])
record_time('repro_s', time.time() - _t_repro)
print('重現核對：Top-5 完全相同 %d/%d 題（最多可以差 %d 題）；重心最大差距 %s（上限 %g）；題目向量最大差距 %s（只記錄）'
      % (_n_identical, _n_q, REPRO_MAX_MISMATCH, _centroid_diff, REPRO_CENTROID_TOL, _query_diff))
print('GPU：這次 %s，prereg01 %s' % (ENV['gpu'], _p01_gpu))
if REPRO['passed']:
    print('重現核對通過')
else:
    print('重現核對沒過：%s。收尾格照樣打包 player_results.zip，然後停下'
          % '、'.join(k for k, v in _criteria.items() if not v))


### seam_p02：seam_val 的逐盤 embedding 和 prereg02 是否逐位元組相同（只記錄）

把 seam_val 的 `cand_games.npz`、`query_games.npz` 和 prereg02 的 dumps 逐個陣列比，記下是否逐位元組相同、emb_sum 的最大差，以及兩次執行的 GPU、torch、cuDNN。這一格只記錄，不會停下；接縫的判定在本機（`tools/t02_player.py run --pool seam_val`，登記第 9 節第 2 條）。

In [ ]:
# seam_p02（test02 新增，只記錄）：seam_val 的逐盤 embedding 和 prereg02 的 dumps 逐個陣列比
_t_seam = time.time()
SEAM_P02 = {'checked_utc': now_utc(), 'smoke': SMOKE, 'reference_dir': os.path.join(P02_RESULT_DIR, 'dumps'),
            'files': {}}
_all_equal, _max_diff = True, 0.0
for _name in ('cand_games.npz', 'query_games.npz'):
    _a = os.path.join(RESULT_DIR, 'seam_val', 'dumps', _name)
    _b = os.path.join(P02_RESULT_DIR, 'dumps', _name)
    with np.load(_a) as _za, np.load(_b) as _zb:
        _keys = sorted(set(_za.files) | set(_zb.files))
        _eq = {_k: bool(_k in _za.files and _k in _zb.files and _za[_k].dtype == _zb[_k].dtype
                        and _za[_k].shape == _zb[_k].shape and np.array_equal(_za[_k], _zb[_k])) for _k in _keys}
        _same_shape = _za['emb_sum'].shape == _zb['emb_sum'].shape
        _d = (float(np.max(np.abs(_za['emb_sum'].astype(np.float64) - _zb['emb_sum'].astype(np.float64))))
              if _same_shape and _za['emb_sum'].size else None)
    SEAM_P02['files'][_name] = {'arrays_equal': _eq, 'bitwise_equal': all(_eq.values()), 'max_abs_emb_diff': _d,
                                'sha256': sha256_file(_a), 'reference_sha256': sha256_file(_b)}
    _all_equal = _all_equal and all(_eq.values())
    _max_diff = None if (_max_diff is None or _d is None) else max(_max_diff, _d)
SEAM_P02.update({
    'bitwise_equal': _all_equal,
    'max_abs_emb_diff': _max_diff,
    'gpu': ENV['gpu'], 'torch': ENV['torch'], 'cudnn': ENV['cudnn'],
    'p02_gpu': P02_ENV.get('gpu') if P02_ENV else None,
    'p02_torch': P02_ENV.get('torch') if P02_ENV else None,
    'p02_cudnn': P02_ENV.get('cudnn') if P02_ENV else None,
    'rule': '只記錄；接縫的判定在本機 tools/t02_player.py（登記第 9 節第 2 條）',
})
save_json('seam_p02.json', SEAM_P02)
update_run(seam_p02_bitwise_equal=_all_equal, seam_p02_max_abs_emb_diff=_max_diff)
record_time('seam_p02_s', time.time() - _t_seam)
print('seam_p02：seam_val 的逐盤 embedding 和 prereg02 %s（emb_sum 最大差 %s）；GPU 這次 %s，prereg02 %s'
      % ('逐位元組相同' if _all_equal else '不完全相同', _max_diff, ENV['gpu'], SEAM_P02['p02_gpu']))


In [ ]:
# 收尾（營運性）：核對三個池的輸出、比對 prereg01／prereg02 前後指紋、記下這次執行；打包；Colab 上把 Drive 的寫入全部送出再卸載
# （test02：重現核對沒過、輸出不對或唯讀目錄有變動時，照樣打包 test02_player_results.zip，最後停下）
FINAL_PROBLEMS = []
for _pool in POOL_ORDER:
    if _pool not in DUMP_INFO_BY_POOL:
        FINAL_PROBLEMS.append('%s 的每盤 embedding 格沒有跑完' % _pool)
        continue
    for _name, _k in (('cand_games.npz', 0), ('query_games.npz', 1)):
        _path = os.path.join(RESULT_DIR, _pool, 'dumps', _name)
        _rows = EXPECTED_COUNTS[_pool][_k]
        if not os.path.exists(_path) or sha256_file(_path) != DUMP_INFO_BY_POOL[_pool][_name]['sha256']:
            FINAL_PROBLEMS.append('%s/dumps/%s 不在或 SHA-256 和寫出時不同' % (_pool, _name))
            continue
        with np.load(_path) as _z:
            if _z['emb_sum'].shape != (_rows, EMBED_DIM) or any(len(_z[_kk]) != _rows for _kk in _z.files if _kk != 'weights_sha256'):
                FINAL_PROBLEMS.append('%s/dumps/%s 的列數不是 %d' % (_pool, _name, _rows))
    _ip = os.path.join(RESULT_DIR, _pool, 'dumps', '_INPUT.json')
    if not os.path.exists(_ip):
        FINAL_PROBLEMS.append('%s/dumps/_INPUT.json 不在' % _pool)
    else:
        with open(_ip, encoding='utf-8') as f:
            if json.load(f) != INPUT_RECORDS[_pool]:
                FINAL_PROBLEMS.append('%s/dumps/_INPUT.json 和這次的輸入檔不同' % _pool)
for _root, _dirs, _files in os.walk(RESULT_DIR):
    for _f in _files:
        if os.path.getsize(os.path.join(_root, _f)) >= 100 * 1024 * 1024:
            FINAL_PROBLEMS.append('%s ≥ 100 MB，結果 zip 會跳過它' % os.path.relpath(os.path.join(_root, _f), RESULT_DIR))
READONLY_AFTER = readonly_fingerprints()
save_json('readonly_after.json', READONLY_AFTER)
_changed = sorted('%s/%s' % (_r, _k) for _r in ('prereg01', 'prereg02')
                  for _k in set(READONLY_BEFORE[_r]['files']) | set(READONLY_AFTER[_r]['files'])
                  if READONLY_BEFORE[_r]['files'].get(_k) != READONLY_AFTER[_r]['files'].get(_k))
READONLY_UNCHANGED = not _changed
if not READONLY_UNCHANGED:
    FINAL_PROBLEMS.append('唯讀目錄有 %d 個檔和開頭不同（例如 %s；見 readonly_before/after.json）'
                          % (len(_changed), _changed[:3]))
if not REPRO['passed']:
    FINAL_PROBLEMS.append('重現核對沒過：%s（見 repro_check.json）'
                          % '、'.join(k for k, v in REPRO['criteria'].items() if not v))
FINAL_STATUS = ('completed' if not FINAL_PROBLEMS else 'readonly_changed' if not READONLY_UNCHANGED
                else 'repro_failed' if not REPRO['passed'] else 'final_check_failed')
record_time('total_s', time.time() - RUN_T0)
set_status(FINAL_STATUS, finished_utc=now_utc(), seconds=time.time() - RUN_T0,
           readonly_unchanged=READONLY_UNCHANGED, readonly_changed_files=_changed, final_problems=FINAL_PROBLEMS)
print(('完成。結果在' if not FINAL_PROBLEMS else '結束，但有問題（見最後的說明）。結果在'), RESULT_DIR)
for _name in sorted(os.listdir(RESULT_DIR)):
    print('  ', _name)

# 結果打包：結果目錄裡除了權重以外的檔案打成一個 zip，放在 results/test02（彩排、SMOKE 是各自的根目錄），方便整包下載
import zipfile
RESULTS_ZIP = os.path.join(RESULT_ROOT, 'test02_player_results.zip')      # test02：固定檔名；成員從 player/ 起算
with zipfile.ZipFile(guard_write(RESULTS_ZIP), 'w', compression=zipfile.ZIP_DEFLATED) as _z:
    for _root, _dirs, _files in os.walk(RESULT_DIR):
        for _f in sorted(_files):
            _p = os.path.join(_root, _f)
            if _f.endswith(('.pth', '.pt', '.tmp')) or os.path.getsize(_p) > 100 * 1024 * 1024:
                continue   # 權重與暫存檔不帶
            _z.write(_p, os.path.relpath(_p, os.path.dirname(RESULT_DIR)))
print('請下載這個檔回傳給 Claude（每池的選擇、融合與交件檔在本機做）：%s（%.1f MB）'
      % (RESULTS_ZIP, os.path.getsize(RESULTS_ZIP) / 1e6))
if IN_COLAB:
    drive.flush_and_unmount()
    print('Drive 已寫入完成並卸載；要再跑請重新「全部執行」')
if FINAL_PROBLEMS:
    raise SystemExit('這次執行沒有通過（狀態 %s），請下載 test02_player_results.zip 並截圖給 Claude：\n%s'
                     % (FINAL_STATUS, '\n'.join('  - ' + p for p in FINAL_PROBLEMS)))
